# Assignment 9: Financial Blockchain Capstone

## African Remittances Payment Prototype

**BLCH9X2 Blockchain, Master of Financial Engineering, University of Johannesburg** (group assignment)

Bukho Ndziweni (220081696), Sizwe November (200819365), Tshireletso Sethaiso (221016661), Bridget Maposa (221034491)

A Sender in South Africa pays in rand and receives a simulated stablecoin, **mZAR**, one for one. The Sender signs a transfer on the chain to a Recipient in Lesotho, who redeems it for loti. Every step goes through a small API (application programming interface), and anyone can check the result against the chain.

| Part | Sections | Contents |
|---|---|---|
| 1. How it is built | 1 to 7 | Setup, then the code, one layer at a time |
| 2. The demonstration | 8 to 16 | Steps i to vi of the brief, then the whole payment in one call |
| 3. Checks | 17 to 18 | Checks on the run, and the rule tests |

**To run:** *Restart Kernel and Run All Cells*. Section 1 installs Flask, requests or ecdsa if any is missing. Every address, signature, transaction identifier, nonce and block hash is the same on any machine. Only the mining seconds differ.

## 1. Setup

Three external libraries: Flask serves the API, requests sends requests to it, and ecdsa signs transactions. Any that are missing are installed into the Python this notebook runs on.

In [1]:
import copy                                # copy.deepcopy makes an independent duplicate of a downloaded chain
import importlib                           # importlib loads a code file, and reloads it after it has been rewritten
import json                                # json prints a transaction in an indented, readable form
import subprocess                          # subprocess runs pip, the program that installs Python libraries
import sys                                 # sys reports which Python is running and where it is installed
from importlib.metadata import PackageNotFoundError, version   # version() reads a library's installed version; the error means it is missing

REQUIRED = {"flask": "Flask", "requests": "requests", "ecdsa": "ecdsa"}   # the three external libraries: installation name and display name
missing = []                                                              # the libraries this Python does not have yet
for package in REQUIRED:                                                  # check each library in turn
    try:                                                                  # asking for a version fails when the library is absent
        version(package)                                                  # succeeds only if the library is installed
    except PackageNotFoundError:                                          # this Python does not have the library
        missing.append(package)                                           # remember it, so it can be installed below
if missing:                                                               # at least one library has to be installed
    print("Installing:", ", ".join(missing))                              # say what is about to happen
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])   # sys.executable is this notebook's own Python, so pip installs into it
    importlib.invalidate_caches()                                         # let Python find the libraries that were just installed

from IPython.display import HTML, display  # shows a web page inside the notebook

print("Python  ", sys.version.split()[0])  # the Python version, for example 3.11.5
for package, label in REQUIRED.items():    # one line per external library
    print(f"{label:<9}", version(package)) # its name and installed version

Python   3.13.9
Flask     3.1.2
requests  2.32.5
ecdsa     0.19.2


# Part 1: How it is built

Six files, written to disk by the cells below. `%%writefile` saves a cell to a file instead of running it, and `%%writefile -a` adds the next piece to the same file.

```
payment_api.py         the API: one endpoint per action
remittance.py          off the chain: custody, reserve, on-ramp, off-ramp, receipts
stablecoin.py          on the chain: the mZAR rules and the ledger
simple_blockchain.py   the chain core from Assignments 1 to 7
verify.py, explorer.py outside parties: recompute everything from the blocks
```

## 2. The chain core: `simple_blockchain.py`

The group's code from Assignments 1 to 7, carried forward. Every change made for the capstone is marked `CAPSTONE` and leaves earlier hashes unchanged.

### 2.1 What the file holds, and hashing (Assignment 1)

The file's map, its imports, and the SHA-256 (Secure Hash Algorithm, 256-bit) helper.

In [2]:
%%writefile simple_blockchain.py
"""
simple_blockchain.py
Assignment 9: Financial Blockchain Capstone (the chain core)
Blockchain module (BLCH9X2), Master of Financial Engineering, University of Johannesburg

WHAT THIS FILE DOES
    Carries the group's SimpleBlockchain build forward from Assignments 1 to 7, in the
    same order as the Assignment 7 notebook:

        Section 1  Assignment 1   SHA-256 (Secure Hash Algorithm, 256-bit) helper
        Section 2  Assignment 2   Block, genesis block, linking
        Section 3  Assignment 3   Transaction and its structural validation
        Section 4  Assignment 4   Wallet: ECDSA (Elliptic Curve Digital Signature Algorithm) keys and signatures
        Section 5  Assignment 5   Blockchain: appending blocks and checking the chain
        Section 6  Assignment 6   Mining (proof-of-work) and the coinbase reward
        Section 7  Assignment 7   Transaction identifiers, confirmations and the fork-choice rule

    The stablecoin rules, the reserve and the payment API (Application Programming
    Interface) are built on top of this file in stablecoin.py, remittance.py and
    payment_api.py.

CAPSTONE EXTENSIONS TO THE EARLIER CODE
    Each one is marked "CAPSTONE" where it appears. All are backward compatible: a
    block or transaction that does not use the new fields hashes exactly as before.

        Transaction   five optional fields (kind, asset, sequence, public_key, reference),
                      included in the dictionary and in the signed bytes only when set
        Wallet        from_seed() for repeatable demonstration keys, and deterministic
                      signatures, so a demonstration prints the same identifiers every run
        Block         from_dict(), so a block can be rebuilt from its network form
        make_coinbase the reward is stamped with the block's time and number
        resolve_fork  the validity test is passed in, so the stablecoin rules are applied
                      before chains are compared

HOW TO RUN
    This file is a library and is not run directly. Run the demonstration with:
        python demo.py
"""

import copy                 # copy.deepcopy makes an independent duplicate of a list or dictionary
import hashlib              # hashlib provides SHA-256, the fingerprint function used throughout
import json                 # json turns Python dictionaries into text in a controlled, repeatable way
import time                 # time measures how long mining takes and supplies default timestamps

from ecdsa import BadSignatureError, SECP256k1, SigningKey, VerifyingKey   # the elliptic curve tools: keys, the Bitcoin curve and the bad-signature error
from ecdsa.util import sigdecode_der, sigencode_der                        # read and write signatures in DER (Distinguished Encoding Rules) form

# ===========================================================================
# SECTION 1: CRYPTOGRAPHY (ASSIGNMENT 1)
# ===========================================================================

def sha256_string(text: str) -> str:
    """Return the SHA-256 fingerprint of a piece of text, written as 64 hexadecimal characters."""
    encoded_text = text.encode("utf-8")               # convert the text into raw bytes, because hashing works on bytes
    return hashlib.sha256(encoded_text).hexdigest()   # hash the bytes and return the result as readable text



Overwriting simple_blockchain.py


### 2.2 Blocks (Assignment 2)

A block is sealed by the hash of its contents, including the hash of the block before it.

In [3]:
%%writefile -a simple_blockchain.py
# ===========================================================================
# SECTION 2: BLOCK (ASSIGNMENT 2)
# ===========================================================================

GENESIS_PREVIOUS_HASH = "0" * 64   # 64 zeros, the agreed convention meaning "there is no earlier block"


class Block:
    """One block in the chain: a batch of transactions, sealed with a fingerprint."""

    def __init__(self, index, timestamp, transactions, previous_hash, nonce=0, difficulty=None):
        """Create a block and immediately seal it by computing its hash."""
        self.index = index                                  # the position of this block in the chain, starting at 0 for genesis
        self.timestamp = int(timestamp)                     # when the block was created, as a Unix timestamp in whole seconds
        self.transactions = copy.deepcopy(transactions)     # an independent copy of the transactions, so later edits elsewhere cannot reach the block
        self.previous_hash = previous_hash                  # the fingerprint of the block before this one, which is what forms the chain
        self.nonce = nonce                                  # the "number used once", the only field a miner changes while searching
        self.difficulty = difficulty                        # Assignment 7: the number of leading zeros this block declares it was mined to
        self.hash = self.compute_hash()                     # seal the block by fingerprinting everything above

    def payload_for_hash(self) -> dict:
        """Collect every field the hash must commit to, leaving out the hash itself."""
        payload = {                                 # the five fields from Assignment 2
            "index": self.index,                    # committing to position stops a block being moved elsewhere in the chain
            "timestamp": self.timestamp,            # committing to time stops the block being back-dated
            "transactions": self.transactions,      # committing to the transactions is what makes the amounts tamper evident
            "previous_hash": self.previous_hash,    # committing to the parent is what links the chain together
            "nonce": self.nonce,                    # committing to the nonce is what makes the mining work count
        }
        if self.difficulty is not None:             # Assignment 7: the difficulty is hashed only when it was declared
            payload["difficulty"] = self.difficulty # so blocks from Assignments 2 to 6 keep the hashes they always had
        return payload                              # hand back the fields to be hashed

    def compute_hash(self) -> str:
        """Recompute this block's fingerprint from its current contents."""
        raw = json.dumps(self.payload_for_hash(), sort_keys=True, separators=(",", ":"))   # one agreed text form: sorted keys and no optional spaces
        return sha256_string(raw)                                                          # fingerprint that text with SHA-256

    def to_dict(self) -> dict:
        """Return the block as a plain dictionary, the form in which it travels over the network."""
        return {**self.payload_for_hash(), "hash": self.hash}   # the hashed fields plus the stored hash, so a reader can check it

    @staticmethod
    def from_dict(data):
        """CAPSTONE: rebuild a block from its network form. Return None if the data is unusable."""
        try:                                                    # data from outside cannot be trusted to have the right shape
            block = Block(                                      # build the block from the hashed fields
                index=int(data["index"]),                       # the claimed position, forced to a whole number
                timestamp=int(data["timestamp"]),               # the claimed time, forced to a whole number
                transactions=list(data["transactions"]),        # the claimed transactions, forced to a list
                previous_hash=str(data["previous_hash"]),       # the claimed parent hash, forced to text
                nonce=int(data["nonce"]),                       # the claimed nonce, forced to a whole number
                difficulty=data.get("difficulty"),              # the declared difficulty, or None if the block declares none
            )
            block.hash = str(data["hash"])                      # keep the hash the sender CLAIMED, so it can be checked later
            return block                                        # hand back the rebuilt block
        except (KeyError, TypeError, ValueError):               # a missing field or a wrong type lands here
            return None                                         # None tells the caller the data was unusable

    def __repr__(self):
        """Control what Python prints when a block is inspected."""
        return f"Block(index={self.index}, hash={self.hash[:10]}...)"   # the position and the start of the hash are enough to recognise it


def create_genesis_block(timestamp=None, transactions=None, difficulty=None) -> Block:
    """Return block 0, the only block that has no parent."""
    return Block(                                       # build the block
        index=0,                                        # genesis always sits at position 0
        timestamp=timestamp or int(time.time()),        # the supplied time, or the current time if none was given
        transactions=transactions or [],                # CAPSTONE: genesis may carry records, such as the stablecoin's charter
        previous_hash=GENESIS_PREVIOUS_HASH,            # 64 zeros, because there is no earlier block
        difficulty=difficulty,                          # CAPSTONE: genesis may declare the difficulty every later block must meet
    )


def create_linked_block(previous: Block, transactions, timestamp=None, nonce=0) -> Block:
    """Return a new block that points at the block before it."""
    return Block(                                       # build the block
        index=previous.index + 1,                       # one position after its parent
        timestamp=timestamp or int(time.time()),        # the supplied time, or the current time if none was given
        transactions=transactions,                      # the transactions this block carries
        previous_hash=previous.hash,                    # the link: the parent's fingerprint is stored inside the child
        nonce=nonce,                                    # the starting nonce, normally 0 before mining
    )


def is_hash_valid(block: Block) -> bool:
    """Return True if the block's stored hash still matches its contents."""
    return block.hash == block.compute_hash()   # any edit after sealing changes the recomputed hash, so the two differ



Appending to simple_blockchain.py


### 2.3 Transactions (Assignment 3)

The five original fields plus five capstone fields (`kind`, `asset`, `sequence`, `public_key`, `reference`). The new fields are signed with the rest and left out when unused.

In [4]:
%%writefile -a simple_blockchain.py
# ===========================================================================
# SECTION 3: TRANSACTIONS (ASSIGNMENT 3)
# ===========================================================================

EXTENSION_FIELDS = ("kind", "asset", "sequence", "public_key", "reference")   # CAPSTONE: the optional fields added to the five from Assignment 3


class Transaction:
    """One payment instruction: sender pays recipient an amount at a moment in time."""

    def __init__(self, sender, recipient, amount, timestamp=None, signature=None, kind=None, asset=None, sequence=None, public_key=None, reference=None):
        """Store the five Assignment 3 fields and the optional capstone fields."""
        self.sender = sender                                                    # the address of the paying party
        self.recipient = recipient                                              # the address of the receiving party
        self.amount = amount                                                    # the value being moved
        self.timestamp = timestamp if timestamp is not None else time.time()    # when the instruction was created (now, if not supplied)
        self.signature = signature                                              # the sender's signature as hexadecimal text, or None before signing
        self.kind = kind                                                        # CAPSTONE: "mint", "transfer" or "burn"
        self.asset = asset                                                      # CAPSTONE: which asset is moved, for example "mZAR"
        self.sequence = sequence                                                # CAPSTONE: how many transactions this sender has made before this one
        self.public_key = public_key                                            # CAPSTONE: the sender's public key, so anyone can check the signature
        self.reference = reference                                              # CAPSTONE: a short reference linking the transaction to an off-chain record

    def to_dict(self) -> dict:
        """Return the transaction as a plain dictionary."""
        data = {                                    # the five fields from Assignment 3, always present
            "sender": self.sender,                  # who is paying
            "recipient": self.recipient,            # who is being paid
            "amount": self.amount,                  # how much
            "timestamp": self.timestamp,            # when it was created
            "signature": self.signature,            # the authorisation
        }
        for field in EXTENSION_FIELDS:              # CAPSTONE: the optional fields
            value = getattr(self, field)            # read the field from this object by name
            if value is not None:                   # a field that was never set is left out
                data[field] = value                 # so a five-field transaction looks exactly as it did in Assignment 3
        return data                                 # hand back the dictionary

    @classmethod
    def from_dict(cls, data):
        """Rebuild a transaction from a dictionary."""
        return cls(                                 # build a new Transaction
            data["sender"],                         # the paying party (required)
            data["recipient"],                      # the receiving party (required)
            data["amount"],                         # the value (required)
            data.get("timestamp"),                  # the creation time
            data.get("signature"),                  # the signature, if there is one
            **{field: data.get(field) for field in EXTENSION_FIELDS},   # CAPSTONE: the optional fields, each None when absent
        )

    def payload_bytes(self) -> bytes:
        """Return the exact bytes a wallet signs: every field except the signature itself."""
        body = {                                    # the four fields signed in Assignment 4
            "amount": self.amount,                  # the value
            "recipient": self.recipient,            # who is being paid
            "sender": self.sender,                  # who is paying
            "timestamp": self.timestamp,            # when it was created
        }
        for field in EXTENSION_FIELDS:              # CAPSTONE: the optional fields are signed too
            value = getattr(self, field)            # read the field from this object by name
            if value is not None:                   # only fields that were set
                body[field] = value                 # so nobody can change the kind, asset, sequence or reference after signing
        return json.dumps(body, sort_keys=True, separators=(",", ":")).encode("utf-8")   # the agreed text form, converted to bytes

    def __repr__(self):
        """Control what Python prints when a transaction is inspected."""
        return f"Transaction(sender={self.sender}, recipient={self.recipient}, amount={self.amount})"   # the three facts that identify a payment


def validate_transaction(tx):
    """Check the shape of a transaction. Return (True, "OK") or (False, the reason)."""
    if not tx.sender or not isinstance(tx.sender, str):                 # the sender must be non-empty text
        return False, "Sender must be a non-empty string"               # otherwise no payer is identified
    if not tx.recipient or not isinstance(tx.recipient, str):           # the recipient must be non-empty text
        return False, "Recipient must be a non-empty string"            # otherwise no beneficiary is identified
    if isinstance(tx.amount, bool) or not isinstance(tx.amount, (int, float)) or tx.amount <= 0:   # CAPSTONE adds the bool test: Python treats True as 1
        return False, "Amount must be positive (>0)"                    # zero, negative and non-numeric amounts are refused
    if isinstance(tx.timestamp, bool) or not isinstance(tx.timestamp, (int, float)):   # the creation time must be a number
        return False, "Timestamp must be numeric"                       # otherwise transactions cannot be ordered in time
    if tx.signature is not None and not isinstance(tx.signature, str):  # a signature, when present, must be text
        return False, "Signature must be a string or None"              # anything else breaks the schema
    return True, "OK"                                                   # every structural rule passed



Appending to simple_blockchain.py


### 2.4 Wallets and signatures (Assignment 4)

An ECDSA (Elliptic Curve Digital Signature Algorithm) key pair. Signatures are now deterministic, so every run prints the same identifiers.

In [5]:
%%writefile -a simple_blockchain.py
# ===========================================================================
# SECTION 4: WALLETS AND DIGITAL SIGNATURES (ASSIGNMENT 4)
# ===========================================================================

def pubkey_to_address(pubkey_bytes: bytes) -> str:
    """Derive an address from a public key: the first 40 hexadecimal characters of its SHA-256 hash."""
    return hashlib.sha256(pubkey_bytes).hexdigest()[:40]   # short enough to read, long enough that two keys never share one


class Wallet:
    """A key pair plus its derived address. Whoever holds the private key controls the address."""

    def __init__(self, private_key: SigningKey, name: str = ""):
        """Build a wallet around an existing private key."""
        self.name = name                                                # a human readable label, used only for display
        self.private_key = private_key                                  # the secret half of the pair: it signs, and must never be shared
        self.public_key = private_key.get_verifying_key()               # the public half, computed from the private key and never the reverse
        self.public_key_hex = self.public_key.to_string().hex()         # the public key as hexadecimal text, safe to publish
        self.address = pubkey_to_address(self.public_key.to_string())   # the address other parties use to pay this wallet

    @classmethod
    def create(cls, name: str = "") -> "Wallet":
        """Create a wallet with a fresh random key. This is the method to use for any real key."""
        private_key = SigningKey.generate(curve=SECP256k1)   # draw a random private key on SECP256k1, the curve Bitcoin uses
        return cls(private_key, name=name)                   # wrap it in a wallet

    @classmethod
    def from_seed(cls, seed: str, name: str = "") -> "Wallet":
        """CAPSTONE: create a wallet whose key is derived from a text seed, so a demonstration is repeatable.

        Anyone who knows the seed can rebuild the private key, so this is for demonstrations only."""
        digest = hashlib.sha256(seed.encode("utf-8")).digest()                      # hash the seed into 32 bytes
        exponent = int.from_bytes(digest, "big") % (SECP256k1.order - 1) + 1        # turn the bytes into a valid private key number, never zero
        private_key = SigningKey.from_secret_exponent(exponent, curve=SECP256k1)    # build the private key from that number
        return cls(private_key, name=name)                                          # wrap it in a wallet

    def sign(self, message: bytes) -> str:
        """Sign a message with the private key. Return the signature as hexadecimal text."""
        signature = self.private_key.sign_deterministic(   # CAPSTONE: the deterministic method of RFC 6979 (an internet standard)
            message,                                       # the bytes being signed
            hashfunc=hashlib.sha256,                       # the message is hashed with SHA-256 before signing
            sigencode=sigencode_der,                       # write the signature in DER form, as in Assignment 4
        )                                                  # the same key and message always give the same signature, and no random number can leak the key
        return signature.hex()                             # hexadecimal text survives being placed in JSON (JavaScript Object Notation)

    @staticmethod
    def verify(message: bytes, sig_hex: str, pubkey_hex: str) -> bool:
        """Check a signature against a message and a public key. No private key is needed."""
        try:                                                                                    # malformed input raises an error rather than returning False
            public_key = VerifyingKey.from_string(bytes.fromhex(pubkey_hex), curve=SECP256k1)   # rebuild the public key from its hexadecimal text
            signature = bytes.fromhex(sig_hex)                                                  # convert the signature back into raw bytes
            return public_key.verify(signature, message, hashfunc=hashlib.sha256, sigdecode=sigdecode_der)   # True only if this key signed these exact bytes
        except (BadSignatureError, ValueError, TypeError, AssertionError):                      # a wrong signature or unusable input lands here
            return False                                                                        # either way the signature is not accepted

    def sign_transaction(self, tx: Transaction) -> None:
        """Sign a transaction in place, filling in its signature field."""
        tx.signature = self.sign(tx.payload_bytes())   # sign the agreed bytes and store the result on the transaction

    def __repr__(self):
        """Control what Python prints when a wallet is inspected. The private key is never shown."""
        return f"Wallet(name={self.name!r}, address={self.address})"   # the label and address only



Appending to simple_blockchain.py


### 2.5 Chain checks and mining (Assignments 5 and 6)

The four integrity pillars, proof-of-work mining, and the miner's reward.

In [6]:
%%writefile -a simple_blockchain.py
# ===========================================================================
# SECTION 5: CHAIN INTEGRITY (ASSIGNMENT 5)
# ===========================================================================

class Blockchain:
    """An ordered list of blocks, each pointing at the one before it."""

    def __init__(self, genesis=None):
        """Start a chain from the supplied genesis block, or from a new one."""
        self.chain = [genesis if genesis is not None else create_genesis_block()]   # the list of blocks, oldest first

    def tip(self) -> Block:
        """Return the newest block, which the next block must point at."""
        return self.chain[-1]   # "[-1]" means the last item in a list

    def append_block(self, block: Block) -> None:
        """Add a block to the chain, refusing it if it does not fit on the tip."""
        tip = self.tip()                                                                                    # the block the new one must extend
        if block.index != tip.index + 1:                                                                    # gate 1: positions must count up by one
            raise ValueError(f"Block {block.index}: index must be {tip.index + 1} to extend the tip.")      # the block is out of sequence
        if block.previous_hash != tip.hash:                                                                 # gate 2: the block must point at the tip
            raise ValueError(f"Block {block.index}: previous_hash does not match the tip's hash.")          # the link is broken
        if not is_hash_valid(block):                                                                        # gate 3: the stored hash must match the contents
            raise ValueError(f"Block {block.index}: stored hash does not match its own computed hash.")     # something was edited after sealing
        self.chain.append(block)                                                                            # all three gates passed, so the block joins the chain

    def make_next_block(self, transactions, timestamp=None, nonce: int = 0) -> Block:
        """Build (but do not append) the block that would follow the tip."""
        return create_linked_block(self.tip(), transactions, timestamp=timestamp, nonce=nonce)   # linked to the current tip

    def verify_chain_verbose(self, difficulty=None):
        """Check the pillars of chain integrity. Return (True, []) or (False, a list of problems)."""
        problems = []                                                           # one readable message per broken rule
        if not self.chain:                                                      # an empty chain cannot be valid
            return False, ["Chain is empty."]                                   # nothing further can be checked
        genesis = self.chain[0]                                                 # block 0 has rules of its own
        if genesis.index != 0:                                                  # the first block must sit at position 0
            problems.append(f"Genesis: index pillar failed (index={genesis.index}, expected 0).")   # record the failure
        if genesis.previous_hash != GENESIS_PREVIOUS_HASH:                      # and must point at the 64 zero sentinel
            problems.append("Genesis: previous_hash sentinel failed (expected 64 zeros).")          # record the failure
        if not is_hash_valid(genesis):                                          # and its stored hash must match its contents
            problems.append("Genesis: self-hash pillar failed (stored hash does not match the contents).")   # record the failure
        prefix = ("0" * difficulty) if difficulty else None                     # the proof-of-work pattern to look for, if one was asked for
        for position in range(1, len(self.chain)):                              # every later block is checked against its parent
            current, previous = self.chain[position], self.chain[position - 1]  # the block and the one before it
            if current.index != position:                                       # pillar 1: the index must equal the position in the list
                problems.append(f"Block {position}: index pillar failed (index={current.index}, expected {position}).")   # record the failure
            if not is_hash_valid(current):                                      # pillar 2: the stored hash must match the contents
                problems.append(f"Block {position}: self-hash pillar failed (stored={current.hash[:12]}..., recomputed={current.compute_hash()[:12]}...).")   # record the failure
            if current.previous_hash != previous.hash:                          # pillar 3: the block must point at its parent
                problems.append(f"Block {position}: link pillar failed (previous_hash={current.previous_hash[:12]}..., block {position - 1}.hash={previous.hash[:12]}...).")   # record the failure
            if prefix and not current.hash.startswith(prefix):                  # pillar 4: the hash must show proof of work
                problems.append(f"Block {position}: difficulty pillar failed (hash {current.hash[:12]}... lacks {difficulty} leading zeros).")   # record the failure
        return len(problems) == 0, problems                                     # valid only when no problem was found

# ===========================================================================
# SECTION 6: MINING AND PROOF-OF-WORK (ASSIGNMENT 6)
# ===========================================================================

BLOCK_REWARD = 50       # the fixed number of native coins created for the miner of each block
COINBASE_SENDER = "NETWORK"   # the sender name that marks a reward: nobody is debited, the coins are new


def mine_with_stats(block: Block, difficulty: int):
    """Try nonce values one after another until the block hash starts with enough zeros.

    Return (number of hashes tried, seconds taken)."""
    target = "0" * difficulty                       # the pattern we demand, for example "0000" when difficulty is 4
    block.nonce = 0                                 # every search starts at zero, which makes the result repeatable
    block.hash = block.compute_hash()               # hash the block with that first nonce
    attempts = 1                                    # one hash has been tried so far
    started = time.perf_counter()                   # start a stopwatch so the cost of mining can be reported
    while not block.hash.startswith(target):        # keep going until the hash starts with enough zeros
        block.nonce += 1                            # move to the next nonce
        block.hash = block.compute_hash()           # hash the block again with the new nonce
        attempts += 1                               # count this attempt
    elapsed = time.perf_counter() - started         # stop the stopwatch
    return attempts, elapsed                        # report how many hashes it took and how long


def make_coinbase(miner_address: str, amount=BLOCK_REWARD, timestamp=None, block_index=None) -> dict:
    """Build the special first transaction of a block, which pays the miner in newly created native coins."""
    reference = None if block_index is None else f"reward for block {block_index}"   # CAPSTONE: naming the block makes every reward unique
    reward = Transaction(                           # the reward follows the same schema as every other transaction
        sender=COINBASE_SENDER,                     # no real sender: these coins did not exist before
        recipient=miner_address,                    # the miner's address
        amount=amount,                              # the fixed block reward
        timestamp=timestamp,                        # CAPSTONE: the block's own time, so the reward is identical on every run
        reference=reference,                        # CAPSTONE: which block this reward belongs to
    )
    return reward.to_dict()                         # blocks store transactions as dictionaries



Appending to simple_blockchain.py


### 2.6 Consensus helpers (Assignment 7)

Transaction identifiers, confirmation depth and the fork-choice rule. The rule now takes its validity test from the stablecoin layer.

In [7]:
%%writefile -a simple_blockchain.py
# ===========================================================================
# SECTION 7: CONSENSUS HELPERS (ASSIGNMENT 7)
# ===========================================================================

MIN_DIFFICULTY = 2      # the network rule from Assignment 7: every mined block needs at least 2 leading zeros


def tx_id(tx: dict) -> str:
    """Return the identifier of a transaction: the SHA-256 hash of the bytes that were signed.

    The signature itself is left out, so re-encoding a signature can never change the identifier."""
    return sha256_string(Transaction.from_dict(tx).payload_bytes().decode("utf-8"))   # fingerprint the signed fields


def confirmations(blocks: list, txid: str) -> int:
    """Return how many blocks confirm a transaction: k = H - h + 1, or 0 if it is not on this chain."""
    height = blocks[-1].index                                       # H, the index of the newest block
    for block in blocks:                                            # search every block
        if any(tx_id(tx) == txid for tx in block.transactions):     # is the transaction inside this one?
            return height - block.index + 1                         # its own block counts as the first confirmation
    return 0                                                        # the transaction is not on this chain


def chain_work(blocks: list, rule: str = "length") -> int:
    """Score a chain: by its number of blocks ("length") or by its cumulative proof-of-work ("work")."""
    if rule == "length":                                            # the longest-chain rule
        return len(blocks)                                          # the score is simply the number of blocks
    if rule == "work":                                              # the heaviest-chain rule
        return sum(16 ** (block.difficulty if block.difficulty is not None else MIN_DIFFICULTY)   # each extra zero makes a block 16 times harder
                   for block in blocks[1:])                         # summed over every mined block (genesis is not mined)
    raise ValueError(f"unknown rule {rule!r}")                      # any other rule name is a programming mistake


def resolve_fork(chains: list, is_valid, rule: str = "length", current=None) -> list:
    """Choose between competing chains: validate first, then compare. Return a copy of the winner.

    CAPSTONE: is_valid is supplied by the caller, so the stablecoin rules decide validity."""
    valid = [chain for chain in chains if is_valid(chain)]              # an invalid chain never wins, whatever its length
    if not valid:                                                       # every candidate broke a rule
        raise ValueError("No valid chain to choose from.")              # there is nothing to adopt
    best = max(chain_work(chain, rule) for chain in valid)              # the highest score among the valid chains
    tied = [chain for chain in valid if chain_work(chain, rule) == best]   # every chain that reaches that score
    if current is not None:                                             # the first-seen tie-break from Assignment 7
        for chain in tied:                                              # look for the chain this node already holds
            if chain[-1].hash == current[-1].hash:                      # the node's own chain is among the best
                return copy.deepcopy(chain)                             # so it keeps the chain it saw first
    winner = min(tied, key=lambda chain: chain[-1].hash)                # otherwise the smallest tip hash wins, so the choice is deterministic
    return copy.deepcopy(winner)                                        # an independent copy, so the caller cannot alter the original


Appending to simple_blockchain.py


## 3. The stablecoin rules: `stablecoin.py`

What the chain itself enforces about mZAR.

### 3.1 Settings

One asset, mZAR, backed one for one by rand. Blocks are mined at difficulty 4.

In [8]:
%%writefile stablecoin.py
"""
stablecoin.py
Assignment 9: Financial Blockchain Capstone (the stablecoin rules and the ledger)
Blockchain module (BLCH9X2), Master of Financial Engineering, University of Johannesburg

WHAT THIS FILE DOES
    Turns the SimpleBlockchain core (simple_blockchain.py) into a ledger for one
    simulated stablecoin, mZAR, where 1 mZAR represents 1 South African rand (ZAR)
    held in reserve.

        make_genesis        block 0, which carries the charter naming the reserve agent
        State               the account table: balances, sequence numbers and supply
        transaction_error   every rule a stablecoin transaction must pass
        replay              validates a whole chain from genesis and rebuilds the State
        Ledger              one node's chain plus its pool of pending transactions

    Three kinds of transaction move mZAR:

        mint       the reserve agent creates mZAR for a customer (the on-ramp)
        transfer   a holder sends mZAR to another address
        burn       a holder destroys mZAR, addressed to the reserve agent (the off-ramp)

THE ACCOUNT MODEL
    Balances are not stored in any block. They are worked out by replaying every
    transaction from genesis, so any party holding the chain reaches the same
    balances. Each address also has a sequence number, the count of transactions it
    has sent, and a transaction is valid only if it carries the next number. That
    stops a signed transaction being submitted a second time.

THE TWO ASSETS
    mZAR is created only by a mint and destroyed only by a burn. The mining reward is
    paid in a separate native coin, SBC, which cannot be transferred or redeemed and
    has no claim on the reserve. Mining therefore never changes the mZAR supply.

HOW TO RUN
    This file is a library and is not run directly. Run the demonstration with:
        python demo.py
"""

import copy                 # copy.deepcopy makes an independent duplicate of a list or dictionary
import re                   # re lets us check that an address has exactly the expected characters

from simple_blockchain import (     # the chain core carried forward from Assignments 1 to 7
    BLOCK_REWARD,                   # the native coins paid to the miner of each block
    COINBASE_SENDER,                # the sender name that marks a reward
    MIN_DIFFICULTY,                 # the lowest proof-of-work difficulty the network allows
    Block,                          # one block
    Blockchain,                     # the list of blocks with its integrity checks
    Transaction,                    # one transaction
    Wallet,                         # used here only for its signature check
    confirmations,                  # how many blocks confirm a transaction
    create_genesis_block,           # builds block 0
    make_coinbase,                  # builds the miner's reward
    mine_with_stats,                # the proof-of-work search
    pubkey_to_address,              # derives an address from a public key
    resolve_fork,                   # the fork-choice rule
    tx_id,                          # the identifier of a transaction
    validate_transaction,           # the structural checks from Assignment 3
)

# ===========================================================================
# CONFIGURATION
# ===========================================================================

ASSET = "mZAR"                          # the stablecoin: one unit represents one rand held in reserve
NATIVE_ASSET = "SBC"                    # SimpleBlockchain coin, the native unit that pays miners
RESERVE_RULE = "1 mZAR = 1 ZAR held in reserve"      # the one-for-one reserve rule, written into the genesis block
GENESIS_TIMESTAMP = 1_790_000_000       # the fixed genesis time used since Assignment 7, so every run shares one genesis
DEMO_DIFFICULTY = 4                     # leading zeros required of every block hash: about 16**4 = 65,536 hashes per block
KINDS = ("mint", "transfer", "burn")    # the only kinds of stablecoin transaction
MAX_AMOUNT = 1_000_000_000_000          # an upper limit on one transaction, so absurd values are refused early
MAX_REFERENCE_LENGTH = 64               # the longest reference a transaction may carry
ADDRESS_PATTERN = re.compile(r"[0-9a-f]{40}")   # an address is exactly 40 lowercase hexadecimal characters

# ===========================================================================
# SMALL CHECKS USED BY THE RULES
# ===========================================================================

def is_address(value) -> bool:
    """Return True if the value has the form of an address derived from a public key."""
    return isinstance(value, str) and ADDRESS_PATTERN.fullmatch(value) is not None   # text, and exactly 40 hexadecimal characters


def is_whole_number(value) -> bool:
    """Return True if the value is a whole number. True and False are excluded, although Python counts them as 1 and 0."""
    return isinstance(value, int) and not isinstance(value, bool)   # an integer that is not a boolean



Overwriting stablecoin.py


### 3.2 Key idea: the genesis block names who may mint

Block 0 carries a charter with the reserve agent's address. Its hash commits to that address, so the mint authority cannot be changed quietly.

In [9]:
%%writefile -a stablecoin.py
# ===========================================================================
# THE GENESIS BLOCK AND ITS CHARTER
# ===========================================================================

def make_charter(issuer_address: str, timestamp: int) -> dict:
    """Build the charter: the genesis record that names the reserve agent, the only address allowed to mint."""
    charter = Transaction(                  # the charter follows the same schema as every other record on the chain
        sender=COINBASE_SENDER,             # it comes from the network's rules, not from an account
        recipient=issuer_address,           # the reserve agent who receives the authority to mint
        amount=0,                           # no value moves: the charter only grants authority
        timestamp=timestamp,                # the genesis time
        kind="charter",                     # marks this record as the charter
        asset=ASSET,                        # the asset the charter governs
        reference=RESERVE_RULE,             # the reserve rule in words, committed to by the genesis hash
    )
    return charter.to_dict()                # blocks store records as dictionaries


def make_genesis(issuer_address: str, difficulty: int = DEMO_DIFFICULTY) -> Block:
    """Build block 0. Its hash commits to the reserve agent's address and to the mining difficulty."""
    return create_genesis_block(                                        # the Assignment 2 genesis builder
        timestamp=GENESIS_TIMESTAMP,                                    # the fixed genesis time
        transactions=[make_charter(issuer_address, GENESIS_TIMESTAMP)], # the charter is the only record in block 0
        difficulty=difficulty,                                          # the difficulty every later block must declare and meet
    )


def charter_issuer(genesis: Block):
    """Read the reserve agent's address from a genesis block. Return None if the charter is missing or altered."""
    records = genesis.transactions                                          # everything block 0 carries
    if len(records) != 1 or not isinstance(records[0], dict):               # there must be exactly one record
        return None                                                         # anything else is not a valid charter
    issuer = records[0].get("recipient")                                    # the address the charter names
    if not is_address(issuer):                                              # it must have the form of an address
        return None                                                         # otherwise there is no usable reserve agent
    if records[0] != make_charter(issuer, genesis.timestamp):               # rebuild the only charter the rules allow for that address
        return None                                                         # any extra or altered field invalidates it
    return issuer                                                           # the reserve agent's address



Appending to stablecoin.py


### 3.3 The account table

Balances are never stored. They are rebuilt by replaying every transaction from genesis, so everyone holding the chain gets the same answer.

In [10]:
%%writefile -a stablecoin.py
# ===========================================================================
# THE ACCOUNT TABLE
# ===========================================================================

class State:
    """Who holds what. It is never stored in a block: it is rebuilt by replaying the chain."""

    def __init__(self, issuer: str):
        """Start with nobody holding anything."""
        self.issuer = issuer        # the reserve agent's address, read from the genesis charter
        self.balances = {}          # address -> mZAR held
        self.rewards = {}           # address -> SBC earned by mining
        self.sequences = {}         # address -> number of transactions that address has sent
        self.minted = 0             # total mZAR ever created
        self.burned = 0             # total mZAR ever destroyed
        self.seen = set()           # the identifier of every transaction already applied

    @property
    def supply(self) -> int:
        """Return the mZAR in circulation."""
        return self.minted - self.burned    # everything created minus everything destroyed

    def copy(self) -> "State":
        """Return an independent duplicate, so a trial change can be thrown away."""
        return copy.deepcopy(self)          # duplicate every table inside the state

    def apply(self, tx: dict) -> None:
        """Move the value of one transaction that has ALREADY passed transaction_error."""
        sender, recipient, amount = tx["sender"], tx["recipient"], tx["amount"]   # the three facts that move value
        if tx["kind"] == "mint":                                                # the reserve agent creates new mZAR
            self.balances[recipient] = self.balances.get(recipient, 0) + amount # credit the customer
            self.minted += amount                                               # the supply grows
        elif tx["kind"] == "transfer":                                          # a holder pays another address
            self.balances[sender] = self.balances.get(sender, 0) - amount       # debit the sender
            self.balances[recipient] = self.balances.get(recipient, 0) + amount # credit the recipient
        else:                                                                   # the only remaining kind is a burn
            self.balances[sender] = self.balances.get(sender, 0) - amount       # debit the holder, and credit nobody
            self.burned += amount                                               # the supply shrinks
        self.sequences[sender] = self.sequences.get(sender, 0) + 1              # the sender has now sent one more transaction
        self.seen.add(tx_id(tx))                                                # remember the identifier so it can never be applied again



Appending to stablecoin.py


### 3.4 Key idea: the gates every transaction must pass

Shape, then authorship (key and signature), then replay (sequence number), then authority (who may mint or burn), then funds.

In [11]:
%%writefile -a stablecoin.py
# ===========================================================================
# THE RULES FOR ONE TRANSACTION
# ===========================================================================

def transaction_error(data, state: State):
    """Check one stablecoin transaction against the current state.

    Return None if it is valid, otherwise (code, message). The code is one of
    "malformed", "bad_signature", "replay", "unauthorised" or "insufficient_funds"."""
    # ----- shape: the Assignment 3 schema plus the capstone fields -----
    if not isinstance(data, dict):                                              # a transaction must be a set of named fields
        return "malformed", "a transaction must be a JSON object"               # JSON is JavaScript Object Notation
    try:                                                                        # a missing required field raises an error
        tx = Transaction.from_dict(data)                                        # read the fields into a Transaction
    except KeyError as missing:                                                 # sender, recipient or amount was absent
        return "malformed", f"the transaction is missing the field {missing}"   # name the missing field
    if tx.to_dict() != data:                                                    # writing it back out must give exactly what came in
        return "malformed", "the transaction has a missing timestamp, an unknown field or a field set to null"   # so nothing unsigned can ride along
    structurally_valid, reason = validate_transaction(tx)                       # the structural checks from Assignment 3
    if not structurally_valid:                                                  # for example an empty sender or a negative amount
        return "malformed", reason                                              # pass on the reason
    if tx.kind not in KINDS:                                                    # the kind decides which rules apply below
        return "malformed", "'kind' must be \"mint\", \"transfer\" or \"burn\"" # anything else is refused
    if tx.asset != ASSET:                                                       # this ledger moves one asset only
        return "malformed", f"'asset' must be \"{ASSET}\""                      # the native coin cannot be transferred
    if not is_whole_number(tx.amount) or tx.amount > MAX_AMOUNT:                # whole units only, so no rounding can ever occur
        return "malformed", f"'amount' must be a whole number from 1 to {MAX_AMOUNT:,}"   # decimals and absurd values are refused
    if not is_whole_number(tx.timestamp):                                       # whole seconds, so the signed text is identical on every machine
        return "malformed", "'timestamp' must be a whole number of seconds"     # decimals are refused
    if not is_whole_number(tx.sequence) or tx.sequence < 0:                     # the sender's transaction count
        return "malformed", "'sequence' must be a whole number, zero or more"   # it is required for every stablecoin transaction
    if not is_address(tx.sender) or not is_address(tx.recipient):               # both parties must be real addresses
        return "malformed", "'sender' and 'recipient' must be 40 character hexadecimal addresses"   # names are not accepted
    if tx.reference is not None and (not isinstance(tx.reference, str) or not tx.reference.isprintable()   # an optional reference must be printable text
                                     or not 1 <= len(tx.reference) <= MAX_REFERENCE_LENGTH):   # and short
        return "malformed", f"'reference' must be 1 to {MAX_REFERENCE_LENGTH} printable characters"   # explain the rule that failed
    if not isinstance(tx.public_key, str) or not isinstance(tx.signature, str):  # both are needed to check authorship
        return "malformed", "a 'public_key' and a 'signature' are required"     # an unsigned transaction is refused
    # ----- authorship: the Assignment 4 signature, now checked by the ledger itself -----
    try:                                                                        # the public key must be valid hexadecimal
        public_key_bytes = bytes.fromhex(tx.public_key)                         # convert it to raw bytes
    except ValueError:                                                          # it contained characters that are not hexadecimal
        return "malformed", "'public_key' must be hexadecimal text"             # so no address can be derived from it
    if pubkey_to_address(public_key_bytes) != tx.sender:                        # the key must be the one the sender's address was derived from
        return "bad_signature", "the public key does not belong to the sender's address"   # somebody else's key cannot spend this account
    if not Wallet.verify(tx.payload_bytes(), tx.signature, tx.public_key):      # the signature must cover these exact fields
        return "bad_signature", "the signature does not match the transaction's contents"   # a field was changed after signing, or the key is wrong
    # ----- replay protection -----
    if tx_id(data) in state.seen:                                               # the same signed instruction was applied before
        return "replay", "this transaction is already on the ledger"            # it cannot move value twice
    expected = state.sequences.get(tx.sender, 0)                                # the next sequence number this sender may use
    if tx.sequence != expected:                                                 # too low means reused, too high means a gap
        return "replay", f"'sequence' is {tx.sequence} but the sender's next sequence number is {expected}"   # say which number is expected
    # ----- authority: who may do what -----
    if tx.kind == "mint" and tx.sender != state.issuer:                         # only the address named in the genesis charter creates mZAR
        return "unauthorised", "only the reserve agent named in the genesis block may mint"   # anyone else is refused
    if tx.kind == "burn" and tx.recipient != state.issuer:                      # a burn is a redemption request to whoever holds the reserve
        return "unauthorised", "a burn must be addressed to the reserve agent"  # so it must name the reserve agent
    if tx.kind == "transfer" and tx.sender == tx.recipient:                     # paying yourself moves nothing
        return "malformed", "'sender' and 'recipient' must be different"        # so it is refused
    # ----- funds -----
    held = state.balances.get(tx.sender, 0)                                     # what the sender holds at this point
    if tx.kind in ("transfer", "burn") and held < tx.amount:                    # nobody may spend or burn more than they hold
        return "insufficient_funds", f"the sender holds {held:,} {ASSET} but the transaction needs {tx.amount:,}"   # say how much is short
    return None                                                                 # None means every rule passed



Appending to stablecoin.py


### 3.5 Checking a block and a whole chain

The reward first, then every transaction in order. `replay` checks a chain from genesis and rebuilds the account table.

In [12]:
%%writefile -a stablecoin.py
# ===========================================================================
# THE RULES FOR ONE BLOCK AND FOR A WHOLE CHAIN
# ===========================================================================

def block_rule_error(block: Block, state: State):
    """Check the transactions of one block. Return a message describing the first problem, or None.

    When the block is valid, state is updated to include it."""
    records = block.transactions                                                # everything the block carries
    if not records or not isinstance(records[0], dict) or records[0].get("sender") != COINBASE_SENDER:   # the Assignment 7 rule
        return "first transaction must be the coinbase"                         # every block starts with the miner's reward
    reward = records[0]                                                         # the reward
    if not is_address(reward.get("recipient")):                                 # it must be paid to a usable address
        return "coinbase recipient is not a valid address"                      # refuse a reward with a broken recipient
    if reward != make_coinbase(reward["recipient"], BLOCK_REWARD, block.timestamp, block.index):   # rebuild the only reward the rules allow
        return "coinbase is not the reward the rules allow"                     # a miner tried to overpay itself or reuse a reward
    working = state.copy()                                                      # work on a copy, so a rejected block changes nothing
    working.rewards[reward["recipient"]] = working.rewards.get(reward["recipient"], 0) + BLOCK_REWARD   # credit the miner in the native coin
    working.seen.add(tx_id(reward))                                             # remember the reward's identifier
    for record in records[1:]:                                                  # every other record must be a stablecoin transaction
        if isinstance(record, dict) and record.get("sender") == COINBASE_SENDER:   # a second reward in the same block
            return "extra coinbase (only one may create coins)"                 # is refused, as in Assignment 7
        problem = transaction_error(record, working)                            # apply every transaction rule, in block order
        if problem is not None:                                                 # one bad transaction invalidates the whole block
            return f"invalid transaction ({problem[1]})"                        # report which rule failed
        working.apply(record)                                                   # move the value before checking the next one
    state.__dict__.update(working.__dict__)                                     # the block is valid, so publish the new state
    return None                                                                 # None means the block passed every rule


def replay(blocks: list, genesis_hash=None, trail=None):
    """Validate a whole chain from genesis and rebuild the account table.

    Return (state, None) if the chain is valid, or (None, a message describing the first problem).
    If trail is a list, one line is added to it for every block that passed."""
    if not blocks or any(block is None for block in blocks):                    # an empty chain or an unreadable block is refused
        return None, "Chain is empty or holds an unreadable block."             # nothing further can be checked
    genesis = blocks[0]                                                         # block 0 sets the rules for the rest
    if genesis_hash is not None and genesis.hash != genesis_hash:               # the caller may insist on one particular genesis
        return None, "Genesis: does not match the expected genesis block."      # this chain belongs to a different network
    issuer = charter_issuer(genesis)                                            # who may mint, according to block 0
    if issuer is None:                                                          # the charter is missing or was altered
        return None, "Genesis: does not carry a valid stablecoin charter."      # so no mint on this chain can be trusted
    required = genesis.difficulty                                               # the difficulty block 0 declares for the network
    if not is_whole_number(required) or required < MIN_DIFFICULTY:             # it must be at least the Assignment 7 minimum
        return None, f"Genesis: declared difficulty must be a whole number of at least {MIN_DIFFICULTY}."   # refuse a network with no real work
    chain = Blockchain(genesis=genesis)                                         # wrap the blocks in the Assignment 5 class
    chain.chain = list(blocks)                                                  # so its integrity check can be reused
    intact, problems = chain.verify_chain_verbose(difficulty=MIN_DIFFICULTY)    # the four pillars: index, self-hash, link and proof of work
    if not intact:                                                              # a block was edited, moved or never mined
        return None, problems[0]                                                # report the first broken pillar
    state = State(issuer)                                                       # start from nobody holding anything
    if trail is not None:                                                       # the caller asked for a line per block
        trail.append({"block": 0, "hash": genesis.hash, "transactions": 1, "signatures": 0,   # block 0 holds the charter and no signatures
                      "result": f"charter names reserve agent {issuer}"})       # say what block 0 established
    for block in blocks[1:]:                                                    # every mined block, oldest first
        declared = block.difficulty                                             # the difficulty this block says it was mined to
        if not is_whole_number(declared) or declared < required:                # it may not be easier than the network's rule
            return None, f"Block {block.index}: declared difficulty {declared} is below the network's {required}."   # an under-mined block is refused
        if not block.hash.startswith("0" * declared):                           # and the hash must really show that much work
            return None, f"Block {block.index}: declared difficulty {declared} is not met by the hash."   # the declaration was false
        problem = block_rule_error(block, state)                                # the reward, signatures, sequence numbers and balances
        if problem is not None:                                                 # the first invalid block ends the replay
            return None, f"Block {block.index}: {problem}"                      # say which block failed and why
        if trail is not None:                                                   # the caller asked for a line per block
            trail.append({"block": block.index, "hash": block.hash, "transactions": len(block.transactions),   # how many records it holds
                          "signatures": len(block.transactions) - 1,            # every record except the reward carries a signature
                          "result": "hash, link, proof of work, reward, signatures, sequences and balances all hold"})   # what was checked
    return state, None                                                          # the chain is valid: hand back the rebuilt account table



Appending to stablecoin.py


### 3.6 The ledger: reading it

One node's chain plus its pool of pending transactions.

In [13]:
%%writefile -a stablecoin.py
# ===========================================================================
# THE LEDGER: ONE NODE'S CHAIN PLUS ITS PENDING POOL
# ===========================================================================

class Ledger:
    """The chain of blocks held by one node, and the transactions waiting to be mined."""

    def __init__(self, issuer_address: str, difficulty: int = DEMO_DIFFICULTY):
        """Start a ledger holding only the genesis block."""
        self.difficulty = difficulty                                            # the proof-of-work difficulty this node mines at
        self.blockchain = Blockchain(genesis=make_genesis(issuer_address, difficulty))   # the Assignment 5 chain, started from our genesis
        self.pending = []                                                       # transactions accepted but not yet mined, oldest first

    @property
    def chain(self) -> list:
        """Return the list of blocks, oldest first."""
        return self.blockchain.chain            # the list held by the Assignment 5 class

    @property
    def genesis_hash(self) -> str:
        """Return the hash of block 0, which identifies this network."""
        return self.chain[0].hash               # the fingerprint of the genesis block

    # -----------------------------------------------------------------------
    # Reading the ledger
    # -----------------------------------------------------------------------
    def state(self) -> State:
        """Replay the chain and return the confirmed account table."""
        state, problem = replay(self.chain)                         # this node's own chain is valid by construction
        if problem is not None:                                     # reaching here would mean the node's memory was corrupted
            raise RuntimeError(f"this node's chain is invalid: {problem}")   # stop loudly instead of serving wrong balances
        return state                                                # the confirmed balances, sequence numbers and supply

    def state_with_pending(self) -> State:
        """Return the account table as it will be once every pending transaction is mined."""
        state = self.state()                                        # start from the confirmed table
        for tx in self.pending:                                     # then apply the waiting transactions in arrival order
            state.apply(tx)                                         # each one passed the rules when it was accepted
        return state                                                # what the next block will produce

    def account(self, address: str) -> dict:
        """Report what one address holds and which sequence number it must use next."""
        confirmed, expected = self.state(), self.state_with_pending()   # the table now, and the table after the pending pool
        return {                                                    # a small report
            "address": address,                                     # whose account this is
            "asset": ASSET,                                         # the asset the balances are in
            "confirmed": confirmed.balances.get(address, 0),        # mZAR already mined into the chain
            "after_pending": expected.balances.get(address, 0),     # mZAR once the pending pool is mined
            "next_sequence": expected.sequences.get(address, 0),    # the number the address must put on its next transaction
            "mining_rewards": confirmed.rewards.get(address, 0),    # SBC earned by mining, which is not redeemable
        }

    def find_transaction(self, txid: str):
        """Look a transaction up by identifier. Return where it is, or None if this node has never seen it."""
        for block in self.chain:                                                # search the mined blocks first
            for position, tx in enumerate(block.transactions):                  # and every record inside each
                if tx_id(tx) == txid:                                           # found it in the chain
                    return {                                                    # report where it was mined
                        "txid": txid,                                           # the identifier that was asked for
                        "status": "confirmed",                                  # it is part of the chain
                        "block_index": block.index,                             # the block that holds it
                        "block_hash": block.hash,                               # that block's fingerprint
                        "position_in_block": position,                          # its place in the block, where 0 is the reward
                        "confirmations": confirmations(self.chain, txid),       # k = H - h + 1, from Assignment 7
                        "transaction": tx,                                      # the transaction itself
                    }
        for tx in self.pending:                                                 # otherwise search the pending pool
            if tx_id(tx) == txid:                                               # found it waiting
                return {"txid": txid, "status": "pending", "block_index": None, "block_hash": None,   # it has no block yet
                        "position_in_block": None, "confirmations": 0, "transaction": tx}             # and no confirmations
        return None                                                             # this node has never seen that identifier



Appending to stablecoin.py


### 3.7 The ledger: changing it

A transaction joins the pool only if it passes the same rules a block is checked against. Mining seals the pool into the next block.

In [14]:
%%writefile -a stablecoin.py
    # -----------------------------------------------------------------------
    # Changing the ledger
    # -----------------------------------------------------------------------
    def submit(self, data) -> tuple:
        """Try to place a signed transaction in the pending pool. Return (code, message, txid)."""
        problem = transaction_error(data, self.state_with_pending())    # the same rules a block is checked against
        if problem is not None:                                         # the transaction broke a rule
            return problem[0], problem[1], None                         # say which, and store nothing
        self.pending.append(copy.deepcopy(data))                        # keep an independent copy in the pool
        return "accepted", "transaction accepted into the pending pool", tx_id(data)   # report success and the identifier

    def mine(self, miner_address: str, timestamp: int) -> tuple:
        """Mine one block holding the reward and every pending transaction. Return (block, attempts, seconds)."""
        index = self.blockchain.tip().index + 1                                 # the new block takes the next position
        reward = make_coinbase(miner_address, BLOCK_REWARD, timestamp, index)   # the miner's reward, in the native coin
        block = self.blockchain.make_next_block([reward] + self.pending, timestamp=timestamp)   # the reward first, then the pool in arrival order
        block.difficulty = self.difficulty                                      # declare the difficulty, as in Assignment 7
        attempts, seconds = mine_with_stats(block, self.difficulty)             # the proof-of-work search from Assignment 6
        self.blockchain.append_block(block)                                     # the Assignment 5 gates: index, link and self-hash
        self.pending = []                                                       # its transactions are no longer pending
        return block, attempts, seconds                                         # report the block and what it cost

    def consider_chain(self, blocks: list) -> tuple:
        """Compare another chain with ours and adopt it if the fork-choice rule says so. Return (adopted, message)."""
        _, problem = replay(blocks, self.genesis_hash)                          # validate before comparing, as in Assignment 7
        if problem is not None:                                                 # an invalid chain never wins, whatever its length
            return False, f"the other chain is invalid ({problem})"             # say why it was refused
        winner = resolve_fork([self.chain, blocks],                             # the two candidates
                              is_valid=lambda chain: replay(chain, self.genesis_hash)[1] is None,   # validity includes every stablecoin rule
                              current=self.chain)                               # ties keep the chain this node saw first
        if winner[-1].hash == self.blockchain.tip().hash:                       # our own chain won or tied
            return False, "the other chain is not longer"                       # nothing changes
        on_winner = {tx_id(tx) for block in winner for tx in block.transactions}   # every identifier on the chain being adopted
        returned = [tx for block in self.chain for tx in block.transactions[1:] # transactions from the chain being abandoned
                    if tx_id(tx) not in on_winner]                              # that the new chain does not already hold
        candidates = returned + self.pending                                    # they go back to the pool ahead of what was already waiting
        self.blockchain.chain = winner                                          # adopt the longer valid chain
        self.pending = []                                                       # rebuild the pool from nothing
        for tx in candidates:                                                   # offer each candidate back in its original order
            self.submit(tx)                                                     # the normal rules drop anything no longer valid
        return True, "adopted the longer valid chain"                           # report the switch


Appending to stablecoin.py


## 4. The payment platform: `remittance.py`

Everything that happens off the chain, and how it is tied to the chain.

### 4.1 Settings and clocks

The corridor is South Africa to Lesotho: the loti is pegged to the rand at par, so 1 mZAR = 1 ZAR = 1 LSL. A logical clock keeps every run repeatable.

In [15]:
%%writefile remittance.py
"""
remittance.py
Assignment 9: Financial Blockchain Capstone (the payment platform)
Blockchain module (BLCH9X2), Master of Financial Engineering, University of Johannesburg

WHAT THIS FILE DOES
    Holds everything that happens off the chain, and connects it to the ledger in
    stablecoin.py. The corridor modelled is South Africa to Lesotho: a sender pays in
    rand (ZAR), and the recipient is paid out in loti (LSL), which is pegged to the
    rand at one for one, so no exchange rate is needed.

        LogicalClock        a repeatable clock, so every run prints the same identifiers
        ReserveBook         the reserve agent's cash book for the money backing mZAR
        RemittancePlatform  wallets and custody, the on-ramp, the off-ramp and receipts

THE THREE LAYERS
    Ledger        the Ledger object: blocks, signatures, balances (stablecoin.py)
    Custody       who holds each private key. A "hosted" wallet's key is held by the
                  platform, which signs on the customer's instruction. A "self" wallet's
                  key never reaches the platform: its owner signs and submits a finished
                  transaction.
    Verification  everything the platform claims can be checked against the chain by
                  an outsider (verify.py), and each receipt is signed by the reserve agent.

THE RESERVE RULE
    Rand is recorded in the reserve BEFORE mZAR is minted, and mZAR is burned and
    confirmed BEFORE loti is paid out. The reserve therefore never falls below the
    mZAR in circulation.

HOW TO RUN
    This file is a library and is not run directly. Run the demonstration with:
        python demo.py
"""

import json                 # json writes a receipt in one agreed text form before it is signed
import threading            # threading provides a lock, so two requests cannot change the platform at the same moment
import time                 # time supplies real timestamps when the wall clock is chosen

from simple_blockchain import Transaction, Wallet, pubkey_to_address, tx_id   # transactions, key pairs, address derivation and transaction identifiers
from stablecoin import ASSET, DEMO_DIFFICULTY, GENESIS_TIMESTAMP, MAX_AMOUNT, NATIVE_ASSET, RESERVE_RULE, Ledger, is_address, is_whole_number   # the stablecoin layer

# ===========================================================================
# CONFIGURATION
# ===========================================================================

DEPOSIT_CURRENCY = "ZAR"            # what the sender pays in: South African rand
PAYOUT_CURRENCY = "LSL"             # what the recipient is paid in: Lesotho loti, pegged to the rand at one for one
CONFIRMATIONS_REQUIRED = 1          # how many blocks must confirm a mint or burn before the platform treats it as final
CLOCK_STEP_SECONDS = 60             # the logical clock moves forward one minute each time it is read
MAX_NAME_LENGTH = 32                # the longest wallet name the platform will accept

# ===========================================================================
# CLOCKS
# ===========================================================================

class LogicalClock:
    """A clock that starts at the genesis time and moves forward a fixed step each time it is read.

    Timestamps then depend only on the order of events, so every run prints the same identifiers."""

    def __init__(self, start: int = GENESIS_TIMESTAMP, step: int = CLOCK_STEP_SECONDS):
        """Set the starting time and the size of each step."""
        self.current = start        # the last time handed out
        self.step = step            # how far the clock moves on each reading

    def now(self) -> int:
        """Move the clock forward one step and return the new time."""
        self.current += self.step   # advance by one step
        return self.current         # the new time, as a Unix timestamp


class WallClock:
    """The real time of day. Identifiers then differ from run to run."""

    def now(self) -> int:
        """Return the current time as a Unix timestamp in whole seconds."""
        return int(time.time())     # the number of seconds since 1 January 1970



Overwriting remittance.py


### 4.2 The reserve cash book and signed receipts

The cash book stands in for the bank account behind mZAR. Receipts are signed by the reserve agent.

In [16]:
%%writefile -a remittance.py
# ===========================================================================
# THE RESERVE
# ===========================================================================

class ReserveBook:
    """The reserve agent's cash book: every rand received and every loti paid out, with a running balance.

    It stands in for the bank account that backs the stablecoin. It lives off the chain."""

    def __init__(self):
        """Start with an empty book."""
        self.entries = []           # every deposit and payout, oldest first
        self.balance = 0            # the money currently held in reserve, in rand

    def record(self, kind: str, amount: int, customer: str, reference: str, txid) -> dict:
        """Add one entry. Deposits carry a positive amount and payouts a negative one."""
        self.balance += amount                          # update the running balance
        entry = {                                       # the line written in the book
            "entry": len(self.entries) + 1,             # a running number
            "type": kind,                               # "deposit" or "payout"
            "amount": amount,                           # positive in, negative out
            "currency": DEPOSIT_CURRENCY,               # the reserve is held in rand
            "customer": customer,                       # the customer's address
            "reference": reference,                     # the conversion or receipt this entry belongs to
            "txid": txid,                               # the on-chain transaction it matches
            "balance": self.balance,                    # the balance after this entry
        }
        self.entries.append(entry)                      # keep the entry
        return entry                                    # hand it back to the caller

# ===========================================================================
# RECEIPTS
# ===========================================================================

def receipt_bytes(receipt: dict) -> bytes:
    """Return the exact bytes the reserve agent signs for a receipt: every field except the signature."""
    unsigned = {name: value for name, value in receipt.items() if name != "signature"}   # copy the fields, leaving the signature out
    return json.dumps(unsigned, sort_keys=True, separators=(",", ":")).encode("utf-8")   # the same agreed text form used for transactions


def receipt_signature_valid(receipt: dict, issuer_address: str) -> bool:
    """Return True if the receipt was signed by the reserve agent named in the genesis block."""
    try:                                                                    # a receipt from outside may be missing fields
        public_key = receipt["issuer_public_key"]                           # the key the receipt claims to be signed with
        if pubkey_to_address(bytes.fromhex(public_key)) != issuer_address:  # that key must belong to the reserve agent's address
            return False                                                    # otherwise anybody could have signed it
        return Wallet.verify(receipt_bytes(receipt), receipt["signature"], public_key)   # the signature must cover these exact fields
    except (KeyError, TypeError, ValueError):                               # a missing or malformed field lands here
        return False                                                        # an unreadable receipt is not accepted



Appending to remittance.py


### 4.3 The platform and its refusals

Every refusal from the ledger is passed on with its own status code.

In [17]:
%%writefile -a remittance.py
# ===========================================================================
# THE PLATFORM
# ===========================================================================

class PlatformError(Exception):
    """Raised when a request cannot be carried out. It carries an HTTP (HyperText Transfer Protocol) status code."""

    def __init__(self, status: int, message: str):
        """Remember the status code and the message."""
        super().__init__(message)       # let Python store the message in the usual way
        self.status = status            # the status code the API should answer with
        self.message = message          # the sentence explaining what went wrong


STATUS_FOR_CODE = {                     # how each ledger outcome is reported over HTTP
    "malformed": 400,                   # 400 means "bad request": the fields were wrong
    "bad_signature": 401,               # 401 means "unauthorised": the request did not prove who sent it
    "unauthorised": 403,                # 403 means "forbidden": the sender is known but may not do this
    "replay": 409,                      # 409 means "conflict": the transaction clashes with what the ledger already holds
    "insufficient_funds": 422,          # 422 means "understood, but cannot be processed"
}


class RemittancePlatform:
    """The payment platform: a ledger, a key store, a reserve book and the two conversion desks."""

    def __init__(self, difficulty: int = DEMO_DIFFICULTY, seed=None, clock=None, confirmations_required: int = CONFIRMATIONS_REQUIRED):
        """Create the operator's own wallets, then a ledger whose genesis block names the reserve agent."""
        self.lock = threading.RLock()                               # a re-entrant lock: one request at a time, and a method may call another
        self.seed = seed                                            # None for random keys, or text for repeatable demonstration keys
        self.clock = clock if clock is not None else LogicalClock() # the repeatable clock unless another was supplied
        self.confirmations_required = confirmations_required        # how deep a mint or burn must be before it is treated as final
        self.keys = {}                                              # address -> Wallet, for every key the platform holds
        self.wallets = {}                                           # address -> the public record of each known wallet
        self.reserve_agent = self._create_key("Reserve agent", "operator")   # issues and redeems mZAR, and signs receipts
        self.validator = self._create_key("Validator", "operator")  # mines blocks and receives the native-coin reward
        self.ledger = Ledger(self.reserve_agent.address, difficulty)   # the chain, with the reserve agent written into block 0
        self.reserve = ReserveBook()                                # the cash book for the money backing mZAR
        self.conversions = []                                       # one record per on-ramp deposit
        self.receipts = []                                          # one receipt per off-ramp redemption
        self.payments = []                                          # one record per payment run through pay()



Appending to remittance.py


### 4.4 Key idea: custody

A *hosted* wallet's key is held by the platform, which signs on instruction. A *self* wallet registers only its address and signs on its owner's device.

In [18]:
%%writefile -a remittance.py
    # -----------------------------------------------------------------------
    # Wallets and custody
    # -----------------------------------------------------------------------
    def _create_key(self, name: str, custody: str) -> Wallet:
        """Create a key pair that the platform itself holds, and record the wallet."""
        if self.seed is None:                                           # normal operation
            wallet = Wallet.create(name)                                # a fresh random key
        else:                                                           # demonstration mode
            wallet = Wallet.from_seed(f"{self.seed}/{name}", name)      # a key derived from the seed and the name, identical on every run
        self.keys[wallet.address] = wallet                              # the platform keeps the private key
        self.wallets[wallet.address] = {                                # the public record, which never includes the private key
            "name": name,                                               # the label shown in the explorer
            "address": wallet.address,                                  # where the wallet is paid
            "custody": custody,                                         # "operator" for the platform's own wallets, "hosted" for customers
            "public_key": wallet.public_key_hex,                        # safe to publish
        }
        return wallet                                                   # hand the wallet back to the caller

    def register_wallet(self, name, address=None) -> dict:
        """Register a customer. With no address the platform creates and holds the key; with an address the customer keeps it."""
        with self.lock:                                                                     # one request at a time
            if not isinstance(name, str) or not name.strip() or len(name) > MAX_NAME_LENGTH or not name.isprintable():   # a short printable name is required
                raise PlatformError(400, f"'name' must be 1 to {MAX_NAME_LENGTH} printable characters")   # explain the rule that failed
            name = name.strip()                                                             # remove any spaces around the name
            if any(record["name"] == name for record in self.wallets.values()):             # names identify customers, so each is used once
                raise PlatformError(409, f"a wallet named '{name}' is already registered")  # refuse the repeat
            if address is None:                                                             # no address supplied: a hosted wallet
                wallet = self._create_key(name, "hosted")                                   # the platform creates and keeps the key
                return dict(self.wallets[wallet.address])                                   # return a copy of the public record
            if not is_address(address):                                                     # an address supplied: it must have the right form
                raise PlatformError(400, "'address' must be a 40 character hexadecimal address")   # explain the rule that failed
            if address in self.wallets:                                                     # each address is registered once
                raise PlatformError(409, "that address is already registered")              # refuse the repeat
            self.wallets[address] = {"name": name, "address": address, "custody": "self", "public_key": None}   # the customer keeps the key
            return dict(self.wallets[address])                                              # return a copy of the public record

    def _customer(self, address, must_be_hosted: bool = False) -> dict:
        """Return the record of a registered customer, or raise an error explaining why the request is refused."""
        record = self.wallets.get(address) if isinstance(address, str) else None            # look the address up
        if record is None or record["custody"] == "operator":                               # unknown, or one of the platform's own wallets
            raise PlatformError(403, "that address is not a registered customer; register the wallet first")   # the know-your-customer gate
        if must_be_hosted and record["custody"] != "hosted":                                # the platform can only sign for keys it holds
            raise PlatformError(403, "the platform does not hold that wallet's key; sign the transaction yourself and send it to POST /transactions")   # point to the other route
        return record                                                                       # the customer's public record

    @staticmethod
    def _amount(value) -> int:
        """Return the amount if it is a whole number in the permitted range, or raise an error."""
        if not is_whole_number(value) or not 1 <= value <= MAX_AMOUNT:                      # decimals, text, booleans, negatives and absurd values are refused
            raise PlatformError(400, f"'amount' must be a whole number from 1 to {MAX_AMOUNT:,}")   # explain the rule that failed
        return value                                                                        # the amount is usable

    def _sign_and_submit(self, signer: Wallet, recipient: str, amount: int, kind: str, reference=None) -> dict:
        """Build a stablecoin transaction, sign it with a key the platform holds, and submit it to the ledger."""
        tx = Transaction(                                                       # the unsigned transaction
            sender=signer.address,                                              # the account being debited (or the reserve agent, for a mint)
            recipient=recipient,                                                # the account being credited
            amount=amount,                                                      # how much
            timestamp=self.clock.now(),                                         # when, on the platform's clock
            kind=kind,                                                          # "mint", "transfer" or "burn"
            asset=ASSET,                                                        # always mZAR
            sequence=self.ledger.account(signer.address)["next_sequence"],      # the signer's next sequence number
            public_key=signer.public_key_hex,                                   # so anyone can check the signature
            reference=reference,                                                # the link to an off-chain record, if any
        )
        signer.sign_transaction(tx)                                             # the private key signs every field above
        code, message, txid = self.ledger.submit(tx.to_dict())                  # the ledger applies every rule before accepting
        if code != "accepted":                                                  # the ledger refused it
            raise PlatformError(STATUS_FOR_CODE[code], message)                 # pass the ledger's reason on
        return {"txid": txid, "transaction": tx.to_dict()}                      # the identifier and the signed transaction



Appending to remittance.py


### 4.5 Key idea: rand in before coins out

The on-ramp records the deposit in the reserve first, and only then signs the mint.

In [19]:
%%writefile -a remittance.py
    # -----------------------------------------------------------------------
    # The on-ramp: rand in, mZAR out
    # -----------------------------------------------------------------------
    def on_ramp(self, address, amount, bank_reference=None) -> dict:
        """Record a rand deposit in the reserve, then mint the same number of mZAR to the customer."""
        with self.lock:                                                                 # one request at a time
            customer = self._customer(address)                                          # only registered customers may deposit
            amount = self._amount(amount)                                               # a whole number greater than zero
            if bank_reference is not None and (not isinstance(bank_reference, str) or len(bank_reference) > 64):   # the bank's own reference is optional
                raise PlatformError(400, "'bank_reference' must be text of at most 64 characters")   # explain the rule that failed
            conversion_id = f"DEP-{len(self.conversions) + 1:04d}"                      # a running identifier such as DEP-0001
            entry = self.reserve.record("deposit", amount, address, conversion_id, None)   # step 1: the rand is in the reserve
            minted = self._sign_and_submit(self.reserve_agent, address, amount, "mint", conversion_id)   # step 2: the reserve agent mints against it
            entry["txid"] = minted["txid"]                                              # link the cash-book entry to the mint
            conversion = {                                                              # the record of this conversion event
                "id": conversion_id,                                                    # its identifier, also carried on the chain as the mint's reference
                "type": "on-ramp",                                                      # rand in, mZAR out
                "customer": address,                                                    # who deposited
                "customer_name": customer["name"],                                      # the customer's registered name
                "deposited": amount,                                                    # rand received
                "deposit_currency": DEPOSIT_CURRENCY,                                   # ZAR
                "minted": amount,                                                       # mZAR created: the same number, by the reserve rule
                "asset": ASSET,                                                         # mZAR
                "rate": "1 ZAR = 1 mZAR",                                               # the conversion rate, in words
                "bank_reference": bank_reference,                                       # the bank's reference for the deposit, if given
                "txid": minted["txid"],                                                 # the mint transaction
                "status": "pending",                                                    # it becomes "confirmed" once the mint is mined
                "block_index": None,                                                    # filled in when the mint is mined
                "block_hash": None,                                                     # filled in when the mint is mined
            }
            self.conversions.append(conversion)                                         # keep the record
            return dict(conversion)                                                     # return a copy

    # -----------------------------------------------------------------------
    # Transfers
    # -----------------------------------------------------------------------
    def transfer(self, sender, recipient, amount, reference=None) -> dict:
        """Send mZAR from a hosted wallet. The platform signs on the customer's instruction."""
        with self.lock:                                                                 # one request at a time
            self._customer(sender, must_be_hosted=True)                                 # the platform must hold the sender's key
            amount = self._amount(amount)                                               # a whole number greater than zero
            return self._sign_and_submit(self.keys[sender], recipient, amount, "transfer", reference)   # sign with the hosted key and submit

    def submit_signed(self, data) -> dict:
        """Accept a transaction that was signed outside the platform, by a customer who holds their own key."""
        with self.lock:                                                                 # one request at a time
            code, message, txid = self.ledger.submit(data)                              # the ledger applies every rule
            if code != "accepted":                                                      # the ledger refused it
                raise PlatformError(STATUS_FOR_CODE[code], message)                     # pass the ledger's reason on
            return {"txid": txid, "transaction": data}                                  # the identifier and the transaction as stored



Appending to remittance.py


### 4.6 Off-ramp: burn, confirm, then pay

The customer burns mZAR. Only when the burn is mined does the reserve pay out loti and sign a receipt.

In [20]:
%%writefile -a remittance.py
    # -----------------------------------------------------------------------
    # The off-ramp: mZAR in, loti out
    # -----------------------------------------------------------------------
    def off_ramp(self, address, amount) -> dict:
        """Burn mZAR from a hosted wallet. The payout and its receipt follow once the burn is confirmed."""
        with self.lock:                                                                 # one request at a time
            self._customer(address, must_be_hosted=True)                                # the platform must hold the customer's key
            amount = self._amount(amount)                                               # a whole number greater than zero
            burned = self._sign_and_submit(self.keys[address], self.reserve_agent.address, amount, "burn", "redemption")   # the customer's key signs the burn
            burned["status"] = "awaiting confirmation"                                  # nothing is paid out until the burn is in a block
            burned["note"] = f"a receipt is issued and {PAYOUT_CURRENCY} paid out once the burn has {self.confirmations_required} confirmation(s)"   # what happens next
            return burned                                                               # the burn's identifier and status

    def _settle(self, block) -> list:
        """Act on a block that has just become final: confirm its mints and pay out its burns. Return the receipts issued."""
        issued = []                                                                     # receipts created by this block
        for tx in block.transactions[1:]:                                               # every record after the miner's reward
            txid = tx_id(tx)                                                            # the transaction's identifier
            if tx["kind"] == "mint":                                                    # a mint completes an on-ramp conversion
                for conversion in self.conversions:                                     # find the conversion it belongs to
                    if conversion["txid"] == txid:                                      # matched by transaction identifier
                        conversion.update(status="confirmed", block_index=block.index, block_hash=block.hash)   # record where it was mined
            if tx["kind"] == "burn":                                                    # a burn is a redemption that must now be honoured
                issued.append(self._issue_receipt(tx, txid, block))                     # pay out and write the receipt
        return issued                                                                   # the receipts created

    def _issue_receipt(self, tx: dict, txid: str, block) -> dict:
        """Pay out a confirmed burn from the reserve and return a receipt signed by the reserve agent."""
        customer = self.wallets.get(tx["sender"])                                       # the registered record of whoever burned
        known = customer is not None and customer["custody"] != "operator"              # payouts go to registered customers only
        receipt = {                                                                     # the receipt, before signing
            "id": f"RCPT-{len(self.receipts) + 1:04d}",                                 # a running identifier such as RCPT-0001
            "type": "off-ramp",                                                         # mZAR in, loti out
            "customer": tx["sender"],                                                   # who redeemed
            "customer_name": customer["name"] if known else None,                       # the customer's registered name, if any
            "burned": tx["amount"],                                                     # mZAR destroyed
            "asset": ASSET,                                                             # mZAR
            "paid_out": tx["amount"] if known else 0,                                   # loti paid: the same number, because the loti is pegged at par
            "payout_currency": PAYOUT_CURRENCY,                                         # LSL
            "rate": f"1 {ASSET} = 1 {DEPOSIT_CURRENCY} = 1 {PAYOUT_CURRENCY}",          # the conversion rate, in words
            "status": "paid" if known else "held: the burning address is not a registered customer",   # an unknown redeemer is held for review
            "txid": txid,                                                               # the burn transaction
            "block_index": block.index,                                                 # the block that holds the burn
            "block_hash": block.hash,                                                   # that block's fingerprint
            "issued_at": self.clock.now(),                                              # when the receipt was written
            "issuer": self.reserve_agent.address,                                       # the reserve agent's address, as named in the genesis block
            "issuer_public_key": self.reserve_agent.public_key_hex,                     # so anyone can check the signature below
        }
        if known:                                                                       # only a paid receipt moves money
            self.reserve.record("payout", -tx["amount"], tx["sender"], receipt["id"], txid)   # the loti leaves the reserve
        receipt["signature"] = self.reserve_agent.sign(receipt_bytes(receipt))          # the reserve agent signs every field above
        self.receipts.append(receipt)                                                   # keep the receipt
        return dict(receipt)                                                            # return a copy

    # -----------------------------------------------------------------------
    # Mining
    # -----------------------------------------------------------------------
    def mine(self) -> dict:
        """Mine the pending transactions into a block, then settle whichever block has just become final."""
        with self.lock:                                                                 # hold the platform still while the block is mined
            block, attempts, seconds = self.ledger.mine(self.validator.address, self.clock.now())   # the proof-of-work search
            final_index = block.index - self.confirmations_required + 1                 # the block that this new block makes final
            receipts = self._settle(self.ledger.chain[final_index]) if final_index >= 1 else []   # genesis holds nothing to settle
            return {"block": block.to_dict(), "attempts": attempts, "seconds": seconds, "receipts_issued": receipts}   # what happened



Appending to remittance.py


### 4.7 The whole payment in one call, and the reserve report

In [21]:
%%writefile -a remittance.py
    # -----------------------------------------------------------------------
    # One call for the whole payment
    # -----------------------------------------------------------------------
    def pay(self, sender, recipient, amount, redeem: bool = True, bank_reference=None) -> dict:
        """Run a whole remittance between two hosted wallets: on-ramp, transfer and (optionally) off-ramp, mining after each step."""
        with self.lock:                                                                 # one payment at a time, start to finish
            self._customer(sender, must_be_hosted=True)                                 # check everything first, so a payment never stops half way
            self._customer(recipient, must_be_hosted=redeem)                            # the recipient's key is needed only if the platform redeems for them
            amount = self._amount(amount)                                               # a whole number greater than zero
            if sender == recipient:                                                     # paying yourself moves nothing
                raise PlatformError(400, "'sender' and 'recipient' must be different")  # so it is refused
            steps = []                                                                  # the trail of on-chain evidence, one line per step

            def confirm(name: str, txid: str) -> list:
                """Mine until the step is final, then add it to the trail. Return any receipts issued."""
                issued = []                                                             # receipts created while confirming this step
                for _ in range(self.confirmations_required):                            # mine as many blocks as finality needs
                    issued += self.mine()["receipts_issued"]                            # each block may settle earlier transactions
                found = self.ledger.find_transaction(txid)                              # where the transaction ended up
                steps.append({"step": name, "txid": txid, "block_index": found["block_index"],   # the step, its transaction and its block
                              "block_hash": found["block_hash"], "confirmations": found["confirmations"]})   # with the block's hash and depth
                return issued                                                           # hand back any receipts

            conversion = self.on_ramp(sender, amount, bank_reference)                   # step 1: rand in, mZAR minted to the sender
            confirm("on-ramp (mint)", conversion["txid"])                               # mine it
            moved = self.transfer(sender, recipient, amount, "remittance")              # step 2: the sender pays the recipient
            confirm("transfer", moved["txid"])                                          # mine it
            receipt = None                                                              # there is no receipt unless the recipient redeems
            if redeem:                                                                  # step 3 is optional
                burned = self.off_ramp(recipient, amount)                               # the recipient burns the mZAR
                issued = confirm("off-ramp (burn)", burned["txid"])                     # mine it, which triggers the payout
                receipt = next(item for item in issued if item["txid"] == burned["txid"])   # the receipt for this burn
            payment = {                                                                 # the record of the whole payment
                "id": f"PAY-{len(self.payments) + 1:04d}",                              # a running identifier such as PAY-0001
                "sender": sender,                                                       # who paid in
                "recipient": recipient,                                                 # who was paid
                "amount": amount,                                                       # how much, in rand, mZAR and loti alike
                "status": "paid out" if redeem else "delivered as mZAR",                # where the value ended up
                "steps": steps,                                                         # the on-chain evidence for each step
                "conversion_id": conversion["id"],                                      # the on-ramp record
                "receipt_id": receipt["id"] if receipt else None,                       # the off-ramp receipt
            }
            self.payments.append(payment)                                               # keep the record
            return dict(payment)                                                        # return a copy

    # -----------------------------------------------------------------------
    # Reading the platform
    # -----------------------------------------------------------------------
    def reserve_report(self) -> dict:
        """Compare the money in reserve with the mZAR in circulation."""
        with self.lock:                                                                 # one consistent reading
            state = self.ledger.state()                                                 # the confirmed account table, replayed from the chain
            return {                                                                    # the proof-of-reserves report
                "rule": RESERVE_RULE,                                                   # the rule being tested
                "reserve_balance": self.reserve.balance,                                # rand held, from the cash book
                "reserve_currency": DEPOSIT_CURRENCY,                                   # ZAR
                "minted": state.minted,                                                 # mZAR ever created, from the chain
                "burned": state.burned,                                                 # mZAR ever destroyed, from the chain
                "supply": state.supply,                                                 # mZAR in circulation
                "asset": ASSET,                                                         # mZAR
                "fully_backed": self.reserve.balance >= state.supply,                   # the reserve rule holds when the reserve covers the supply
                "surplus": self.reserve.balance - state.supply,                         # rand held beyond the supply, for example a deposit not yet minted
                "entries": [dict(entry) for entry in self.reserve.entries],             # the cash book itself
            }

    def find_record(self, record_id: str):
        """Return the conversion or receipt with this identifier (or this transaction identifier), or None."""
        with self.lock:                                                                 # one consistent reading
            for record in self.conversions + self.receipts:                             # search both lists
                if record_id in (record["id"], record["txid"]):                         # matched by either identifier
                    return dict(record)                                                 # return a copy
            return None                                                                 # nothing matched

    def snapshot(self) -> dict:
        """Return everything the explorer needs in one consistent reading of the platform."""
        with self.lock:                                                                 # nothing may change while the reading is taken
            return {                                                                    # the snapshot
                "asset": ASSET,                                                         # the stablecoin
                "native_asset": NATIVE_ASSET,                                           # the mining-reward coin
                "difficulty": self.ledger.difficulty,                                   # the mining rule
                "genesis_hash": self.ledger.genesis_hash,                               # the fingerprint that identifies this network
                "reserve_agent": self.reserve_agent.address,                            # who may mint
                "chain": [block.to_dict() for block in self.ledger.chain],              # every block in its network form
                "pending": list(self.ledger.pending),                                   # a copy of the pending pool
                "wallets": [dict(record) for record in self.wallets.values()],          # every known wallet's public record
                "reserve": self.reserve_report(),                                       # the proof-of-reserves report
                "conversions": [dict(record) for record in self.conversions],           # every on-ramp record
                "receipts": [dict(record) for record in self.receipts],                 # every off-ramp receipt
            }


Appending to remittance.py


## 5. The independent verifier: `verify.py`

### 5.1 What the verifier does, and how it reads a node

In [22]:
%%writefile verify.py
"""
verify.py
Assignment 9: Financial Blockchain Capstone (the independent verifier)
Blockchain module (BLCH9X2), Master of Financial Engineering, University of Johannesburg

WHAT THIS PROGRAM DOES
    Lets any party check a payment without trusting the platform. It downloads the
    chain from a node, then recomputes everything on this side:

        audit_chain     replays the whole chain: every hash, link, proof of work, reward,
                        signature, sequence number and balance, from genesis
        locate          finds one transaction and reports its block and confirmations
        check_receipt   checks an off-ramp receipt against the chain and the reserve
                        agent's signature

    The only thing taken from the node is the list of blocks. Every verdict printed
    here is worked out locally, so a node that lied about a payment would be caught.

HOW TO RUN (a node must already be running, see payment_api.py or "python demo.py --hold")
    python verify.py                                   audit the chain of the node on port 5000
    python verify.py --txid <transaction id>           also prove one transaction is in the chain
    python verify.py --receipt RCPT-0001               also check one off-ramp receipt
    python verify.py --genesis <hash>                  insist on one particular genesis block
    python verify.py --node http://127.0.0.1:5001      read a different node
"""

import argparse             # argparse reads options such as --txid from the command line

import requests             # requests sends HTTP (HyperText Transfer Protocol) requests to the node

from remittance import receipt_signature_valid                              # the receipt signature check
from simple_blockchain import Block, Transaction, Wallet, tx_id             # blocks, transactions, the signature check and identifiers
from stablecoin import ASSET, replay                                        # the asset name and the full chain validation

# ===========================================================================
# CONFIGURATION
# ===========================================================================

DEFAULT_NODE = "http://127.0.0.1:5000"      # the node that is read unless another is named
REQUEST_TIMEOUT = 120                       # seconds to wait for the node, long enough for a block to be mined

# ===========================================================================
# TALKING TO A NODE
# ===========================================================================

def call_node(method: str, node: str, path: str, payload=None) -> tuple:
    """Send one request to a node. Return (status code, decoded JSON), or (None, None) if it could not be reached."""
    base = node if "//" in node else "http://" + node           # accept "127.0.0.1:5000" as well as a full web address
    try:                                                        # the node may be switched off
        with requests.Session() as session:                     # a session is one short-lived connection manager
            session.trust_env = False                           # ignore any proxy settings: a local node is reached directly
            response = session.request(method, base.rstrip("/") + path, json=payload, timeout=REQUEST_TIMEOUT)   # send the request and wait for the reply
        return response.status_code, response.json()            # hand back the status code and the decoded body
    except (requests.RequestException, ValueError):             # connection problems, timeouts and non-JSON replies land here
        return None, None                                       # tell the caller the node could not be used



Overwriting verify.py


### 5.2 Key idea: checking without trusting the node

Only the blocks are taken from the node. Every hash, link, proof of work, signature, sequence number and balance is recomputed here.

In [23]:
%%writefile -a verify.py
# ===========================================================================
# CHECKS WORKED OUT ON THIS SIDE
# ===========================================================================

def audit_chain(chain: list, genesis_hash=None) -> dict:
    """Replay a chain received as a list of dictionaries and report whether every rule holds."""
    blocks = [Block.from_dict(item) for item in chain] if isinstance(chain, list) else []   # rebuild each block, keeping the hash it CLAIMS
    trail = []                                                              # receives one line per block that passes
    state, problem = replay(blocks, genesis_hash, trail)                    # every rule, from genesis, recomputed here
    return {                                                                # the audit report
        "valid": problem is None,                                           # True only if nothing was wrong
        "problem": problem,                                                 # the first broken rule, or None
        "chain_length": len(blocks),                                        # how many blocks were received
        "blocks_passed": len(trail),                                        # how many passed before any problem
        "signatures_checked": sum(line["signatures"] for line in trail),    # how many signatures were verified
        "genesis_hash": blocks[0].hash if blocks and blocks[0] is not None else None,   # which network this chain belongs to
        "reserve_agent": state.issuer if state else None,                   # who may mint, read from block 0
        "minted": state.minted if state else None,                          # mZAR ever created
        "burned": state.burned if state else None,                          # mZAR ever destroyed
        "supply": state.supply if state else None,                          # mZAR in circulation
        "checks": trail,                                                    # the line for each block
    }


def locate(chain: list, txid: str):
    """Find a transaction in a chain. Return where it is and whether its signature holds, or None if it is absent."""
    for item in chain:                                                      # search every block
        for position, tx in enumerate(item["transactions"]):                # and every record inside it
            if tx_id(tx) != txid:                                           # recompute the identifier rather than trusting a label
                continue                                                    # not this one
            signed = tx.get("signature") is not None                        # rewards and the charter carry no signature
            signature_valid = Wallet.verify(Transaction.from_dict(tx).payload_bytes(), tx["signature"], tx["public_key"]) if signed else None   # check it here
            return {                                                        # the proof of inclusion
                "txid": txid,                                               # the identifier that was asked for
                "block_index": item["index"],                               # the block that holds it
                "block_hash": item["hash"],                                 # the hash that block claims
                "block_hash_recomputed": Block.from_dict(item).compute_hash(),   # the hash worked out here from the block's contents
                "position_in_block": position,                              # its place in the block, where 0 is the reward
                "confirmations": chain[-1]["index"] - item["index"] + 1,    # k = H - h + 1
                "signature_valid": signature_valid,                         # True, False, or None for an unsigned record
                "transaction": tx,                                          # the transaction itself
            }
    return None                                                             # the transaction is not in this chain


def check_receipt(receipt: dict, chain: list, reserve_agent: str) -> dict:
    """Check an off-ramp receipt against the chain. Return one True or False per test."""
    found = locate(chain, receipt.get("txid", ""))                          # the burn the receipt points at
    burn = found["transaction"] if found else {}                            # the burn's fields, or nothing if it is absent
    return {                                                                # one line per test
        "signed_by_reserve_agent": receipt_signature_valid(receipt, reserve_agent),   # the signature belongs to the address in block 0
        "burn_is_on_chain": found is not None and burn.get("kind") == "burn",         # the transaction exists and really is a burn
        "block_matches": found is not None and (found["block_index"], found["block_hash"]) == (receipt.get("block_index"), receipt.get("block_hash")),   # same block
        "customer_matches": burn.get("sender") == receipt.get("customer"),  # the receipt names whoever burned
        "amount_matches": burn.get("amount") == receipt.get("burned"),      # the receipt states the amount that was burned
    }



Appending to verify.py


### 5.3 Printing the verdict

In [24]:
%%writefile -a verify.py
# ===========================================================================
# THE REPORT
# ===========================================================================

def render_audit(audit: dict) -> str:
    """Return the audit report as plain text."""
    lines = ["CHAIN AUDIT (every check recomputed locally from the blocks the node supplied)"]   # the heading
    for line in audit["checks"]:                                            # one line per block that passed
        lines.append(f"  block {line['block']}  {line['hash']}")            # the block and its full hash
        lines.append(f"           {line['transactions']} record(s), {line['signatures']} signature(s) checked: {line['result']}")   # what was checked
    if audit["valid"]:                                                      # nothing was wrong
        lines.append(f"  RESULT: VALID. {audit['chain_length']} blocks, {audit['signatures_checked']} signatures, "   # the summary
                     f"{audit['minted']:,} {ASSET} minted, {audit['burned']:,} burned, {audit['supply']:,} in circulation.")   # with the supply figures
        lines.append(f"  Genesis hash : {audit['genesis_hash']}")           # which network this is
        lines.append(f"  Reserve agent: {audit['reserve_agent']}")          # who may mint
    else:                                                                   # a rule was broken
        lines.append(f"  RESULT: INVALID. {audit['problem']}")           # name the first broken rule and the block it is in
    return "\n".join(lines)                                                 # one string with a line break between lines


def render_inclusion(found, txid: str) -> str:
    """Return the proof of inclusion for one transaction as plain text."""
    if found is None:                                                       # the transaction is not in the chain
        return f"TRANSACTION {txid}\n  NOT FOUND in any block of this chain."   # say so plainly
    tx = found["transaction"]                                               # the transaction's fields
    signature = {True: "valid", False: "INVALID", None: "not applicable (unsigned record)"}[found["signature_valid"]]   # the signature verdict in words
    matches = "matches" if found["block_hash"] == found["block_hash_recomputed"] else "DOES NOT MATCH"   # the block hash verdict in words
    return "\n".join([                                                      # one fact per line
        f"TRANSACTION {txid}",                                              # the identifier
        f"  Contents      : {tx.get('kind', 'reward')} of {tx['amount']:,} from {tx['sender']} to {tx['recipient']}",   # what it does
        f"  Signature     : {signature}",                                   # checked against the public key inside the transaction
        f"  Included in   : block {found['block_index']}, position {found['position_in_block']}",   # where it sits
        f"  Block hash    : {found['block_hash']}",                         # the hash the block claims
        f"  Recomputed    : {found['block_hash_recomputed']} ({matches})",  # the hash worked out here
        f"  Confirmations : {found['confirmations']}",                      # how deep it is
    ])

# ===========================================================================
# RUNNING THE VERIFIER FROM THE COMMAND LINE
# ===========================================================================

def main():
    """Download a node's chain, audit it, and optionally prove one transaction or check one receipt."""
    parser = argparse.ArgumentParser(description="Independently verify a payment against a node's chain.")   # describes the program for --help
    parser.add_argument("--node", default=DEFAULT_NODE, help="address of the node to read (default http://127.0.0.1:5000)")   # which node
    parser.add_argument("--txid", help="a transaction identifier to prove is in the chain")       # optional
    parser.add_argument("--receipt", help="an off-ramp receipt identifier to check, such as RCPT-0001")   # optional
    parser.add_argument("--genesis", help="the genesis hash the chain must start from")           # optional
    arguments = parser.parse_args()                                         # read what was typed
    status, reply = call_node("GET", arguments.node, "/chain")              # the only data taken on trust: the blocks themselves
    if status != 200:                                                       # the node did not answer
        raise SystemExit(f"No node answered at {arguments.node}. Start one with: python payment_api.py")   # explain how to fix it
    chain = reply["chain"]                                                  # the list of blocks
    audit = audit_chain(chain, arguments.genesis)                           # replay everything locally
    print(render_audit(audit))                                              # the audit report
    if not audit["valid"]:                                                  # nothing in an invalid chain can be relied on
        raise SystemExit(1)                                                 # stop with an error status
    if arguments.txid:                                                      # a transaction was named
        print()                                                             # a blank line
        print(render_inclusion(locate(chain, arguments.txid), arguments.txid))   # the proof of inclusion
    if arguments.receipt:                                                   # a receipt was named
        status, receipt = call_node("GET", arguments.node, f"/receipts/{arguments.receipt}")   # fetch the receipt from the node
        if status != 200:                                                   # the node holds no such receipt
            raise SystemExit(f"The node holds no receipt {arguments.receipt}.")   # say so
        print(f"\nRECEIPT {receipt['id']}")                                 # the heading
        for test, passed in check_receipt(receipt, chain, audit["reserve_agent"]).items():   # run each test
            print(f"  {test:<24}: {'yes' if passed else 'NO'}")             # one line per test


if __name__ == "__main__":      # True only when this file is run directly, not when it is imported
    main()                      # run the verifier


Appending to verify.py


## 6. The explorer: `explorer.py`

A view of the node a person can read, as text or as a web page.

### 6.1 What the explorer does, and how it reads a node

In [25]:
%%writefile explorer.py
"""
explorer.py
Assignment 9: Financial Blockchain Capstone (the explorer)
Blockchain module (BLCH9X2), Master of Financial Engineering, University of Johannesburg

WHAT THIS PROGRAM DOES
    Shows what a node holds in a form a person can read, the way a public block
    explorer does. There are two views, each produced as plain text and as an HTML
    (HyperText Markup Language) page:

        the overview         a summary of the node, its blocks, its transactions, the
                             balance of every wallet, the conversions and the reserve
        the payment record   one transaction with its full identifier, its signature
                             check, and the block that holds it

    Balances are not copied from the node. The explorer replays the chain itself
    (verify.py), so what it shows is what any outside party would work out.

    Run from the command line, this program is a client of the API (Application
    Programming Interface) in payment_api.py and reads only its public endpoints.

HOW TO RUN (a node must already be running, see payment_api.py or "python demo.py --hold")
    python explorer.py                                 the overview of the node on port 5000
    python explorer.py --txid <transaction id>         the payment record for one transaction
    python explorer.py --last 10                       show up to 10 rows per table
    python explorer.py --html explorer.html            also save the view as a web page
    python explorer.py --node http://127.0.0.1:5001    read a different node
"""

import argparse                              # argparse reads options such as --txid from the command line
import html                                  # html.escape makes text safe to place inside a web page
from datetime import datetime, timezone      # used to turn a Unix timestamp into a readable date and time

from simple_blockchain import BLOCK_REWARD, COINBASE_SENDER, tx_id   # the reward size, the reward's sender name and transaction identifiers
from verify import DEFAULT_NODE, audit_chain, call_node, locate      # the node reader and the local checks

# ===========================================================================
# CONFIGURATION
# ===========================================================================

DEFAULT_ROWS = 8                # how many recent blocks and transactions the overview lists
HASH_CHARACTERS = 16            # how much of each 64 character hash the overview displays
ADDRESS_CHARACTERS = 10         # how much of an unnamed address the overview displays

# ===========================================================================
# READING A NODE THROUGH ITS API
# ===========================================================================

def fetch_snapshot(node: str = DEFAULT_NODE):
    """Read a node over HTTP (HyperText Transfer Protocol). Return a snapshot, or None if it cannot be reached."""
    replies = {}                                                            # the decoded reply of each endpoint
    for path in ("/chain", "/transactions/pending", "/wallets", "/reserve", "/receipts"):   # the five public endpoints the explorer needs
        status, reply = call_node("GET", node, path)                        # read one endpoint
        if status != 200:                                                   # every one of them must answer
            return None                                                     # otherwise there is nothing reliable to show
        replies[path] = reply                                               # keep the reply
    chain = replies["/chain"]                                               # the chain reply also carries the network's parameters
    return {                                                                # the same shape that RemittancePlatform.snapshot() produces
        "node": node,                                                       # which node was read
        "asset": chain["asset"],                                            # the stablecoin's name
        "native_asset": chain["native_asset"],                              # the mining-reward coin's name
        "difficulty": chain["difficulty"],                                  # the mining rule
        "genesis_hash": chain["genesis_hash"],                              # the fingerprint that identifies the network
        "reserve_agent": chain["reserve_agent"],                            # who may mint
        "chain": chain["chain"],                                            # the blocks
        "pending": replies["/transactions/pending"]["transactions"],        # the pending pool
        "wallets": replies["/wallets"]["wallets"],                          # the known wallets
        "reserve": replies["/reserve"],                                     # the proof-of-reserves report
        "conversions": replies["/receipts"]["conversions"],                 # the on-ramp records
        "receipts": replies["/receipts"]["receipts"],                       # the off-ramp receipts
    }



Overwriting explorer.py


### 6.2 Formatting helpers

In [26]:
%%writefile -a explorer.py
# ===========================================================================
# SMALL FORMATTING HELPERS
# ===========================================================================

def shorten(hash_hex: str) -> str:
    """Show the start of a hash followed by three dots, so tables stay narrow."""
    return hash_hex[:HASH_CHARACTERS] + "..."       # the leading characters are enough to recognise a hash


def format_time(timestamp) -> str:
    """Turn a Unix timestamp into a date and time in UTC (Coordinated Universal Time)."""
    moment = datetime.fromtimestamp(timestamp, tz=timezone.utc)     # interpret the number of seconds as a UTC moment
    return moment.strftime("%Y-%m-%d %H:%M")                        # for example "2026-09-21 21:34"


def wallet_names(snapshot: dict) -> dict:
    """Return a lookup from address to the name it was registered under."""
    names = {record["address"]: record["name"] for record in snapshot["wallets"]}   # every registered wallet
    names[COINBASE_SENDER] = "(new coins)"                          # a reward has no sender, so say where the value came from
    return names                                                    # the finished lookup


def party(address: str, names: dict) -> str:
    """Show an address by its registered name, or by its first characters if it has none."""
    return names.get(address) or address[:ADDRESS_CHARACTERS] + "..."   # a name when there is one, otherwise a shortened address


def describe(tx: dict, snapshot: dict) -> tuple:
    """Return (kind, amount with its unit) for any record found in a block."""
    if tx.get("kind") == "charter":                                         # the genesis record
        return "charter", "-"                                               # it moves no value
    if tx["sender"] == COINBASE_SENDER:                                     # the miner's reward
        return "reward", f"{tx['amount']:,} {snapshot['native_asset']}"     # paid in the native coin
    return tx["kind"], f"{tx['amount']:,} {tx['asset']}"                    # a stablecoin transaction, paid in mZAR



Appending to explorer.py


### 6.3 Turning a reading into rows

The explorer replays the chain itself instead of trusting the node's figures.

In [27]:
%%writefile -a explorer.py
# ===========================================================================
# TURNING A SNAPSHOT INTO ROWS
# ===========================================================================

def transaction_row(place: str, tx: dict, snapshot: dict, names: dict) -> list:
    """Describe one record as a list of table cells."""
    kind, amount = describe(tx, snapshot)                           # what the record does and how much it moves
    txid = tx_id(tx)                                                # its full identifier
    sequence = tx.get("sequence")                                   # only signed stablecoin transactions carry a sequence number
    return [                                                        # one cell per column
        place,                                                      # the block number, or the word "pending"
        kind,                                                       # charter, reward, mint, transfer or burn
        "(genesis)" if kind == "charter" else party(tx["sender"], names),   # who it is from; the charter comes from the genesis rules
        party(tx["recipient"], names),                              # who it is to
        amount,                                                     # the amount with its unit
        "-" if sequence is None else str(sequence),                 # the sender's sequence number, if any
        tx.get("reference") or "-",                                 # the reference, if any
        (shorten(txid), f"/explorer/tx/{txid}"),                    # the start of the identifier, linked to its payment record on the web page
    ]


def build_overview(snapshot: dict, last: int = DEFAULT_ROWS) -> tuple:
    """Reduce a snapshot to the summary facts and the tables that both views display."""
    chain, reserve, names = snapshot["chain"], snapshot["reserve"], wallet_names(snapshot)   # the three things most rows are built from
    asset, native = snapshot["asset"], snapshot["native_asset"]                 # the two asset names
    audit = audit_chain(chain, snapshot["genesis_hash"])                        # replay the chain here, without trusting the node's own figures
    verdict = ("valid: every hash, link, proof of work, signature and balance was recomputed by this explorer"   # the chain passed
               if audit["valid"] else f"INVALID: {audit['problem']}")           # or the first broken rule
    backed = "yes" if reserve["fully_backed"] else "NO"                         # the reserve rule in one word
    facts = [                                                                   # the summary shown at the top
        ("Node", snapshot["node"]),                                             # which node this is
        ("Asset", f"{asset}. {reserve['rule']}"),                               # the stablecoin and its reserve rule
        ("Reserve agent", snapshot["reserve_agent"]),                           # the only address that may mint
        ("Genesis hash", snapshot["genesis_hash"]),                             # the fingerprint that identifies this network
        ("Chain height", f"{len(chain) - 1} ({len(chain)} blocks including genesis)"),   # height counts from zero
        ("Difficulty", f"{snapshot['difficulty']} leading zeros"),              # the mining rule
        ("Chain check", verdict),                                               # the explorer's own verdict
        ("Supply", f"{reserve['supply']:,} {asset} in circulation ({reserve['minted']:,} minted, {reserve['burned']:,} burned)"),   # from the chain
        ("Reserve", f"{reserve['reserve_balance']:,} {reserve['reserve_currency']} held. Fully backed: {backed}"),   # from the cash book
        ("Pending", f"{len(snapshot['pending'])} transaction(s) waiting to be mined"),   # the size of the pool
    ]
    block_rows = [                                                              # one row per recent block
        [str(block["index"]), format_time(block["timestamp"]), str(len(block["transactions"])),   # position, time and number of records
         f"{block['nonce']:,}", shorten(block["hash"]), shorten(block["previous_hash"])]         # the nonce found, and the two hashes
        for block in reversed(chain[-last:])                                    # the last few blocks, newest first
    ]
    mined = [(block["index"], tx) for block in chain for tx in block["transactions"]]   # every mined record with its block number
    transaction_rows = [transaction_row(str(index), tx, snapshot, names) for index, tx in reversed(mined[-last:])]   # the last few, newest first
    pending_rows = [transaction_row("pending", tx, snapshot, names) for tx in snapshot["pending"]]   # one row per waiting transaction
    balances, rewards = {}, {}                                                  # worked out below from the explorer's own replay
    if audit["valid"]:                                                          # balances from an invalid chain would mean nothing
        for index, tx in mined:                                                 # walk every mined record once more
            kind, _ = describe(tx, snapshot)                                    # what the record does
            if kind == "reward":                                                # a reward credits the miner in the native coin
                rewards[tx["recipient"]] = rewards.get(tx["recipient"], 0) + BLOCK_REWARD   # add the reward
            if kind in ("mint", "transfer"):                                    # a mint or transfer credits the recipient
                balances[tx["recipient"]] = balances.get(tx["recipient"], 0) + tx["amount"]   # add the amount
            if kind in ("transfer", "burn"):                                    # a transfer or burn debits the sender
                balances[tx["sender"]] = balances.get(tx["sender"], 0) - tx["amount"]         # subtract the amount
    custody = {record["address"]: record["custody"] for record in snapshot["wallets"]}   # who holds each registered wallet's key
    holders = list(custody) + [address for address in balances if address not in custody]   # registered wallets first, then any other holder
    balance_rows = [[names.get(address, "(unregistered)"), address, custody.get(address, "-"),   # name, full address and custody
                     f"{balances.get(address, 0):,}", f"{rewards.get(address, 0):,}"]            # mZAR held and SBC earned
                    for address in holders]                                     # one row per holder
    conversion_rows = [                                                         # on-ramp records, then off-ramp receipts
        [record["id"], record["type"], record["customer_name"] or party(record["customer"], names),   # identifier, direction and customer
         f"{record['deposited']:,} {record['deposit_currency']}", f"{record['minted']:,} {record['asset']}",   # rand in, mZAR out
         record["status"], "-" if record["block_index"] is None else str(record["block_index"]),              # status and block
         (shorten(record["txid"]), f"/explorer/tx/{record['txid']}")]           # the mint transaction, linked
        for record in snapshot["conversions"]                                   # one row per deposit
    ] + [                                                                       # followed by
        [record["id"], record["type"], record["customer_name"] or party(record["customer"], names),   # identifier, direction and customer
         f"{record['burned']:,} {record['asset']}", f"{record['paid_out']:,} {record['payout_currency']}",   # mZAR in, loti out
         record["status"], str(record["block_index"]),                          # status and block
         (shorten(record["txid"]), f"/explorer/tx/{record['txid']}")]           # the burn transaction, linked
        for record in snapshot["receipts"]                                      # one row per redemption
    ]
    reserve_rows = [[str(entry["entry"]), entry["type"], f"{entry['amount']:+,}", party(entry["customer"], names),   # number, type, signed amount, customer
                     entry["reference"], f"{entry['balance']:,}"]               # the record it belongs to and the running balance
                    for entry in reserve["entries"]]                            # one row per cash-book entry
    transaction_headers = ["Block", "Kind", "From", "To", "Amount", "Seq", "Reference", "Txid"]   # shared by two tables
    sections = [                                                                # each table: a title, column headings and rows
        (f"BLOCKS (newest first, {len(block_rows)} of {len(chain)})",           # the title says how many are shown
         ["Index", "Time (UTC)", "Records", "Nonce", "Hash", "Previous hash"], block_rows),   # headings, then the rows
        (f"TRANSACTIONS (newest first, {len(transaction_rows)} of {len(mined)})", transaction_headers, transaction_rows),   # the mined records
        (f"PENDING ({len(pending_rows)})", transaction_headers, pending_rows),  # the waiting transactions
        (f"WALLET BALANCES (confirmed, replayed from the chain)", ["Name", "Address", "Custody", asset, native], balance_rows),   # who holds what
        ("CONVERSIONS AND RECEIPTS", ["Id", "Type", "Customer", "In", "Out", "Status", "Block", "Txid"], conversion_rows),   # the two desks
        (f"RESERVE CASH BOOK ({reserve['reserve_currency']})", ["Entry", "Type", "Amount", "Customer", "Reference", "Balance"], reserve_rows),   # the money behind the coin
    ]
    return facts, sections                                                      # everything the two views need


def build_payment_record(snapshot: dict, txid: str):
    """Return the facts about one transaction and its block, or None if the chain does not hold it."""
    found = locate(snapshot["chain"], txid)                                     # search the chain, recomputing identifiers
    if found is None:                                                           # the transaction is not in any block
        return None                                                             # there is nothing to show
    tx, names = found["transaction"], wallet_names(snapshot)                    # the transaction and the name lookup
    block = snapshot["chain"][found["block_index"]]                             # the block that holds it
    kind, amount = describe(tx, snapshot)                                       # what it does and how much it moves
    signature = {True: "valid (checked against the public key carried in the transaction)",   # the signature verdict in words
                 False: "INVALID", None: "none (this record is not signed)"}[found["signature_valid"]]   # for all three cases
    sealed = "matches" if found["block_hash"] == found["block_hash_recomputed"] else "DOES NOT MATCH"   # the block hash verdict in words
    return [                                                                    # one (label, value) pair per line
        ("Transaction ID", txid),                                               # the full 64 character identifier
        ("Kind", kind),                                                         # charter, reward, mint, transfer or burn
        ("Amount", amount),                                                     # the amount with its unit
        ("From", "(genesis)" if kind == "charter" else f"{party(tx['sender'], names)}  {tx['sender']}"),   # the sender's name and full address
        ("To", f"{party(tx['recipient'], names)}  {tx['recipient']}"),          # the recipient's name and full address
        ("Sequence", "-" if tx.get("sequence") is None else str(tx["sequence"])),   # the sender's sequence number
        ("Reference", tx.get("reference") or "-"),                              # the link to an off-chain record
        ("Created (UTC)", format_time(tx["timestamp"])),                        # when the transaction was created
        ("Signature", signature),                                               # rechecked by the explorer
        ("Status", f"confirmed, {found['confirmations']} confirmation(s)"),     # how deep it is in the chain
        ("Block", f"{found['block_index']} (position {found['position_in_block']} of {len(block['transactions'])} records; position 0 is the miner's reward)"),   # where it sits
        ("Block hash", found["block_hash"]),                                    # the full hash of its block
        ("Block hash check", f"recomputed from the block's contents: {sealed}"),   # the explorer's own recomputation
        ("Previous block hash", block["previous_hash"]),                        # the link to the block before
        ("Block nonce", f"{block['nonce']:,}"),                                 # the nonce the miner found
        ("Block difficulty", f"{block.get('difficulty', '-')} leading zeros"),  # the difficulty the block declares
        ("Block time (UTC)", format_time(block["timestamp"])),                  # when the block was mined
    ]



Appending to explorer.py


### 6.4 The text views

In [28]:
%%writefile -a explorer.py
# ===========================================================================
# THE TEXT VIEWS
# ===========================================================================

def cell_text(cell) -> str:
    """Return the text of a table cell. A linked cell is a (text, link) pair, and only its text is printed."""
    return cell[0] if isinstance(cell, tuple) else cell     # drop the link, which only the web page uses


def render_text(snapshot: dict, last: int = DEFAULT_ROWS) -> str:
    """Return the overview as plain text for the terminal."""
    facts, sections = build_overview(snapshot, last)                            # the facts and tables to display
    lines = ["=" * 110, f"EXPLORER  node {snapshot['node']}", "=" * 110]        # a heading between two rules
    for name, value in facts:                                                   # the summary, one fact per line
        lines.append(f"{name:<14}: {value}")                                    # "<14" pads the name so the colons line up
    for title, headers, rows in sections:                                       # then each table in turn
        lines.append("")                                                        # a blank line before the table
        lines.append(title)                                                     # the table's title
        if not rows:                                                            # an empty table is shown as one word
            lines.append("  (none)")                                            # so the reader knows nothing is missing
            continue                                                            # move on to the next table
        table = [headers] + [[cell_text(cell) for cell in row] for row in rows] # the headings followed by the rows, as plain text
        widths = [max(len(row[column]) for row in table) for column in range(len(headers))]   # each column is as wide as its longest cell
        for row in table:                                                       # print the headings, then every row
            lines.append("  " + "  ".join(cell.ljust(width) for cell, width in zip(row, widths)).rstrip())   # pad each cell to its column width
    return "\n".join(lines)                                                     # one string with a line break between lines


def render_payment_text(snapshot: dict, txid: str) -> str:
    """Return the payment record for one transaction as plain text."""
    record = build_payment_record(snapshot, txid)                               # the facts to display
    if record is None:                                                          # the chain does not hold the transaction
        return f"PAYMENT RECORD\n  No confirmed transaction has the identifier {txid}"   # say so plainly
    lines = ["PAYMENT RECORD"]                                                  # the heading
    for label, value in record:                                                 # one fact per line
        lines.append(f"  {label:<20}: {value}")                                 # "<20" pads the label so the colons line up
    return "\n".join(lines)                                                     # one string with a line break between lines



Appending to explorer.py


### 6.5 The web page views

In [29]:
%%writefile -a explorer.py
# ===========================================================================
# THE WEB PAGE VIEWS
# ===========================================================================

# The styling rules for the pages, written in CSS (Cascading Style Sheets). The pages are black on white only,
# and every rule starts with .bx so the styling cannot leak into a surrounding page such as a notebook.
# "!important" keeps the tables looking the same when a notebook applies its own table styling.
PAGE_STYLE = """
.bx { font-family: Calibri, Arial, sans-serif; color: #000; background: #fff; max-width: 1150px; margin: 0 auto; padding: 16px; }
.bx h1 { font-size: 22px; margin: 0 0 12px 0; }
.bx h2 { font-size: 15px; margin: 22px 0 6px 0; }
.bx table { border-collapse: collapse; width: 100%; font-size: 13px; table-layout: auto !important; margin: 0 !important; }
.bx th, .bx td { border: 1px solid #000 !important; padding: 4px 8px; text-align: left !important; color: #000; background: #fff; }
.bx th { font-weight: bold; }
.bx td.mono { font-family: Consolas, "Courier New", monospace; overflow-wrap: anywhere; }
.bx a { color: #000; }
.bx p { font-size: 13px; margin: 4px 0; }
"""
MONOSPACE_COLUMNS = ("Hash", "Previous hash", "Txid", "Address", "Value")   # columns that hold hashes or addresses use a fixed-width font


def html_table(headers: list, rows: list) -> str:
    """Return one HTML table. Every cell is escaped, so text from a transaction can never act as page code."""
    if not rows:                                                                # an empty table is shown as one word
        return "<p>(none)</p>"                                                  # so the reader knows nothing is missing
    head = "".join(f"<th>{html.escape(cell)}</th>" for cell in headers)         # the heading cells
    body = ""                                                                   # the rows are added to this text one by one
    for row in rows:                                                            # each row of the table
        cells = ""                                                              # the cells of this row
        for header, cell in zip(headers, row):                                  # each cell together with its column heading
            style = ' class="mono"' if header in MONOSPACE_COLUMNS else ""      # hashes and addresses use a fixed-width font
            if isinstance(cell, tuple):                                         # a linked cell: (text, link)
                content = f'<a href="{html.escape(cell[1])}">{html.escape(cell[0])}</a>'   # the escaped text inside an escaped link
            else:                                                               # an ordinary cell
                content = html.escape(cell)                                     # the escaped text
            cells += f"<td{style}>{content}</td>"                               # add the cell
        body += f"<tr>{cells}</tr>"                                             # close the row
    return f"<table><tr>{head}</tr>{body}</table>"                              # the finished table


def html_page(title: str, parts: list) -> str:
    """Wrap pieces of content in a complete, self-contained web page."""
    content = "\n".join(parts)                                                  # all the pieces, one per line
    return (                                                                    # the complete page
        "<!DOCTYPE html>\n"                                                     # tells the browser this is modern HTML
        '<html lang="en"><head><meta charset="utf-8">\n'                        # the language and the character set
        f"<title>{html.escape(title)}</title>\n"                                # the text shown on the browser tab
        f"<style>{PAGE_STYLE}</style></head>\n"                                 # the styling rules above
        f'<body><div class="bx">\n{content}\n</div></body></html>\n'            # the visible content
    )


def render_html(snapshot: dict, last: int = DEFAULT_ROWS) -> str:
    """Return the overview as a self-contained web page."""
    facts, sections = build_overview(snapshot, last)                            # the same facts and tables as the text view
    parts = [f"<h1>{html.escape(snapshot['asset'])} explorer, node {html.escape(snapshot['node'])}</h1>"]   # the page heading
    parts.append(html_table(["Item", "Value"], [[name, value] for name, value in facts]))   # the summary as a two column table
    for title, headers, rows in sections:                                       # then each table in turn
        parts.append(f"<h2>{html.escape(title)}</h2>")                          # the table's title
        parts.append(html_table(headers, rows))                                 # the table itself
    return html_page(f"{snapshot['asset']} explorer", parts)                    # wrap the pieces in a page


def render_payment_html(snapshot: dict, txid: str) -> str:
    """Return the payment record for one transaction as a self-contained web page."""
    record = build_payment_record(snapshot, txid)                               # the facts to display
    parts = ["<h1>Payment record</h1>"]                                         # the page heading
    if record is None:                                                          # the chain does not hold the transaction
        parts.append(f"<p>No confirmed transaction has the identifier {html.escape(txid)}</p>")   # say so plainly
    else:                                                                       # the transaction was found
        parts.append(html_table(["Item", "Value"], [[label, value] for label, value in record]))   # the facts as a two column table
    parts.append('<p><a href="/explorer">Back to the overview</a></p>')         # a link back to the first page
    return html_page("Payment record", parts)                                   # wrap the pieces in a page

# ===========================================================================
# RUNNING THE EXPLORER FROM THE COMMAND LINE
# ===========================================================================

def main():
    """Read a running node and print the overview or one payment record."""
    parser = argparse.ArgumentParser(description="Print an explorer view of a running node.")   # describes the program for --help
    parser.add_argument("--node", default=DEFAULT_NODE, help="address of the node to read (default http://127.0.0.1:5000)")   # which node
    parser.add_argument("--txid", help="show the payment record for this transaction identifier")   # optional
    parser.add_argument("--last", type=int, default=DEFAULT_ROWS, help="rows per table in the overview (default 8)")   # how much to show
    parser.add_argument("--html", metavar="FILE", help="also save the view as a web page in FILE")   # optional page output
    arguments = parser.parse_args()                                             # read what was typed
    snapshot = fetch_snapshot(arguments.node)                                   # read the node through its API
    if snapshot is None:                                                        # the node did not answer
        raise SystemExit(f"No node answered at {arguments.node}. Start one with: python payment_api.py")   # explain how to fix it
    rows = max(1, arguments.last)                                               # always show at least one row
    if arguments.txid:                                                          # one transaction was asked for
        text, page = render_payment_text(snapshot, arguments.txid), render_payment_html(snapshot, arguments.txid)   # the payment record, both ways
    else:                                                                       # otherwise show everything
        text, page = render_text(snapshot, rows), render_html(snapshot, rows)   # the overview, both ways
    print(text)                                                                 # the text view
    if arguments.html:                                                          # a page was asked for as well
        with open(arguments.html, "w", encoding="utf-8") as page_file:          # open the file for writing
            page_file.write(page)                                               # write the page
        print(f"\nWeb page written to {arguments.html}")                        # say where it went


if __name__ == "__main__":      # True only when this file is run directly, not when it is imported
    main()                      # print the explorer


Appending to explorer.py


## 7. The payment API: `payment_api.py`

Flask, one endpoint per action, bound to `127.0.0.1` so it is reachable from this computer only.

### 7.1 The endpoints at a glance

In [30]:
%%writefile payment_api.py
"""
payment_api.py
Assignment 9: Financial Blockchain Capstone (the payment API)
Blockchain module (BLCH9X2), Master of Financial Engineering, University of Johannesburg

WHAT THIS PROGRAM DOES
    Wraps one RemittancePlatform (see remittance.py) in a REST (Representational State
    Transfer) API (Application Programming Interface) built with Flask. Each endpoint
    is a web address that reads or changes the platform:

        Wallets and custody
            POST /wallets                register a customer wallet (hosted, or self-custody by address)
            GET  /wallets                every known wallet
            GET  /accounts/<address>     the mZAR balance and next sequence number of an address
        The payment flow
            POST /onramp                 deposit rand, mint the same number of mZAR
            POST /transfer               send mZAR from a hosted wallet (the platform signs)
            POST /transactions           submit a transaction signed outside the platform
            POST /offramp                burn mZAR from a hosted wallet, to be paid out in loti
            POST /mine                   mine the pending transactions into a block
            POST /payments               the whole flow in one call: on-ramp, transfer, off-ramp
            GET  /payments/<id>          the record of one such payment
        Records
            GET  /reserve                the reserve cash book against the mZAR in circulation
            GET  /receipts               every conversion record and receipt
            GET  /receipts/<id>          one conversion record or receipt
        Verification
            GET  /chain                  the full chain
            GET  /chain/verify           the node's own audit of its chain
            GET  /transactions/pending   transactions waiting to be mined
            GET  /transactions/<txid>    one transaction, its block and its confirmations
        Explorer
            GET  /explorer               a web page summarising the node
            GET  /explorer/tx/<txid>     a web page for one transaction and its block

    Every reply except the two explorer pages is JSON (JavaScript Object Notation).
    The exact request and reply of every endpoint is documented in README.md.

SAFETY
    The server binds to 127.0.0.1, so it can only be reached from this computer. There
    is no login, and the platform holds private keys in memory, which is acceptable for
    a local demonstration only. Do not change the host to expose it on a network.

HOW TO RUN
    python payment_api.py                       one node on port 5000, with random keys
    python payment_api.py --seed demo           repeatable keys, for demonstrations only
    python payment_api.py --port 5001           listen on another port
    python payment_api.py --difficulty 3        mine with three leading zeros instead of four
    python payment_api.py --wall-clock          stamp records with the real time of day
"""

import argparse                      # argparse reads options such as --port from the command line
import logging                       # logging lets us quieten the web server's request log when running in the background
import socket                        # socket lets us test whether a port is already taken
import threading                     # threading lets a node run in the background while other code carries on

from flask import Flask, Response, jsonify, request   # the pieces of Flask this file uses
from werkzeug.serving import make_server              # builds a web server object that can be stopped again

from explorer import render_html, render_payment_html                       # the two explorer pages
from remittance import PlatformError, RemittancePlatform, WallClock         # the platform, its error type and the real-time clock
from simple_blockchain import MIN_DIFFICULTY                                # the lowest difficulty the network allows
from stablecoin import ASSET, DEMO_DIFFICULTY, NATIVE_ASSET, RESERVE_RULE, is_address   # names, the default difficulty and the address check
from verify import audit_chain                                              # the full chain audit

# ===========================================================================
# CONFIGURATION
# ===========================================================================

HOST = "127.0.0.1"       # the loopback address: reachable from this computer only
DEFAULT_PORT = 5000      # the port a node listens on unless told otherwise

ENDPOINTS = [                                                                           # the list returned by GET /
    "POST /wallets                register a customer wallet",                          # wallets and custody
    "GET  /wallets                every known wallet",                                  # wallets and custody
    "GET  /accounts/<address>     balance and next sequence number of an address",      # wallets and custody
    "POST /onramp                 deposit rand, mint the same number of mZAR",          # the payment flow
    "POST /transfer               send mZAR from a hosted wallet",                      # the payment flow
    "POST /transactions           submit a transaction signed outside the platform",    # the payment flow
    "POST /offramp                burn mZAR from a hosted wallet, to be paid out",      # the payment flow
    "POST /mine                   mine the pending transactions into a block",          # the payment flow
    "POST /payments               the whole flow in one call",                          # the payment flow
    "GET  /payments/<id>          the record of one such payment",                      # the payment flow
    "GET  /reserve                the reserve against the mZAR in circulation",         # records
    "GET  /receipts               every conversion record and receipt",                 # records
    "GET  /receipts/<id>          one conversion record or receipt",                    # records
    "GET  /chain                  the full chain",                                      # verification
    "GET  /chain/verify           the node's own audit of its chain",                   # verification
    "GET  /transactions/pending   transactions waiting to be mined",                    # verification
    "GET  /transactions/<txid>    one transaction, its block and its confirmations",    # verification
    "GET  /explorer               web page summarising the node",                       # explorer
    "GET  /explorer/tx/<txid>     web page for one transaction and its block",          # explorer
]



Overwriting payment_api.py


### 7.2 Status and wallets

In [31]:
%%writefile -a payment_api.py
# ===========================================================================
# BUILDING THE FLASK APPLICATION
# ===========================================================================

def create_app(platform: RemittancePlatform, port: int = DEFAULT_PORT) -> Flask:
    """Create the Flask application that exposes one platform."""
    app = Flask(__name__)                          # the Flask application object
    app.config["PLATFORM"] = platform              # keep a reference so other code can reach the platform
    app.json.sort_keys = False                     # keep reply fields in the order written below
    node_url = f"http://{HOST}:{port}"             # how this node describes itself

    def body() -> dict:
        """Return the JSON object sent with the request, or refuse the request if there is none."""
        data = request.get_json(silent=True)                                    # read the JSON body, or None if it is missing or malformed
        if not isinstance(data, dict):                                          # every POST endpoint expects named fields
            raise PlatformError(400, "the request body must be a JSON object")  # 400 means "bad request"
        return data                                                             # the fields that were sent

    def explorer_view() -> dict:
        """Return a snapshot of the platform, labelled with this node's address."""
        return {"node": node_url, **platform.snapshot()}                        # the explorer shows which node it read

    # -----------------------------------------------------------------------
    # Status
    # -----------------------------------------------------------------------
    @app.route("/", methods=["GET"])               # GET / answers with a short status report
    def index():
        """Describe this node and list its endpoints."""
        view = platform.snapshot()                               # one consistent reading of the platform
        return jsonify({                                         # reply in JSON
            "service": f"{ASSET} remittance prototype",          # what this node is
            "node": node_url,                                    # which node is answering
            "reserve_rule": RESERVE_RULE,                        # the rule the stablecoin is issued under
            "reserve_agent": view["reserve_agent"],              # the only address that may mint
            "genesis_hash": view["genesis_hash"],                # the fingerprint that identifies this network
            "difficulty": view["difficulty"],                    # the mining rule
            "chain_length": len(view["chain"]),                  # how many blocks it holds, including genesis
            "pending_count": len(view["pending"]),               # how many transactions are waiting
            "endpoints": ENDPOINTS,                              # what can be asked of it
        }), 200                                                  # 200 means "OK"

    # -----------------------------------------------------------------------
    # Wallets and custody
    # -----------------------------------------------------------------------
    @app.route("/wallets", methods=["POST"])       # POST /wallets registers a customer
    def create_wallet():
        """Register a wallet. Without an address the platform creates and holds the key."""
        data = body()                                                            # the fields that were sent
        record = platform.register_wallet(data.get("name"), data.get("address")) # create or register the wallet
        holder = "the platform holds the key" if record["custody"] == "hosted" else "the customer holds the key"   # who can sign for it
        return jsonify({"message": f"wallet registered: {holder}", "wallet": record}), 201   # 201 means "created"

    @app.route("/wallets", methods=["GET"])        # GET /wallets lists every known wallet
    def list_wallets():
        """Return the public record of every wallet. Private keys are never included."""
        wallets = platform.snapshot()["wallets"]                                 # the public records
        return jsonify({"count": len(wallets), "wallets": wallets}), 200         # how many, then the records

    @app.route("/accounts/<address>", methods=["GET"])   # GET /accounts/<address> reports one account
    def get_account(address):
        """Return what an address holds and the sequence number it must use next."""
        if not is_address(address):                                              # refuse text that could never be an address
            raise PlatformError(400, "an address is 40 hexadecimal characters")  # 400 means "bad request"
        with platform.lock:                                                      # read the ledger while nothing is changing it
            report = platform.ledger.account(address)                            # confirmed balance, balance after pending, next sequence
            known = platform.wallets.get(address)                                # the registered record, if there is one
        report["name"] = known["name"] if known else None                        # the registered name
        report["custody"] = known["custody"] if known else None                  # who holds the key
        return jsonify(report), 200                                              # the account report



Appending to payment_api.py


### 7.3 The payment endpoints

On-ramp, transfer, signed submission, off-ramp, mining and the one-call payment.

In [32]:
%%writefile -a payment_api.py
    # -----------------------------------------------------------------------
    # The payment flow
    # -----------------------------------------------------------------------
    @app.route("/onramp", methods=["POST"])        # POST /onramp converts rand into mZAR
    def onramp():
        """Record a rand deposit in the reserve and mint the same number of mZAR to the customer."""
        data = body()                                                            # the fields that were sent
        conversion = platform.on_ramp(data.get("address"), data.get("amount"), data.get("bank_reference"))   # deposit, then mint
        return jsonify({                                                         # the conversion event
            "message": f"deposit recorded in the reserve; a mint of {conversion['minted']:,} {ASSET} is waiting to be mined",   # what happened
            "conversion": conversion,                                            # the conversion record
            "reserve_balance": platform.reserve_report()["reserve_balance"],     # the reserve after the deposit
        }), 201                                                                  # 201 means "created"

    @app.route("/transfer", methods=["POST"])      # POST /transfer sends mZAR from a hosted wallet
    def transfer():
        """Sign a transfer with a key the platform holds, and place it in the pending pool."""
        data = body()                                                            # the fields that were sent
        result = platform.transfer(data.get("sender"), data.get("recipient"), data.get("amount"), data.get("reference"))   # sign and submit
        return jsonify({"message": "transfer signed by the platform and accepted into the pending pool", **result}), 201   # 201 means "created"

    @app.route("/transactions", methods=["POST"])  # POST /transactions accepts a transaction signed elsewhere
    def submit_transaction():
        """Check a transaction signed outside the platform and place it in the pending pool."""
        result = platform.submit_signed(body())                                  # the ledger applies every rule
        return jsonify({"message": "signed transaction accepted into the pending pool", **result}), 201   # 201 means "created"

    @app.route("/offramp", methods=["POST"])       # POST /offramp starts a redemption
    def offramp():
        """Burn mZAR from a hosted wallet. The payout follows when the burn is confirmed."""
        data = body()                                                            # the fields that were sent
        result = platform.off_ramp(data.get("address"), data.get("amount"))      # sign and submit the burn
        return jsonify({"message": "burn accepted into the pending pool", **result}), 202   # 202 means "accepted, not yet completed"

    @app.route("/mine", methods=["POST"])          # POST /mine mines one block
    def mine():
        """Mine a block holding every pending transaction, then settle what it made final."""
        result = platform.mine()                                                 # the proof-of-work search happens here
        return jsonify({                                                         # the block was mined
            "message": f"block {result['block']['index']} mined",                # a sentence saying so
            "block": result["block"],                                            # the new block
            "attempts": result["attempts"],                                      # how many hashes the search needed
            "seconds": round(result["seconds"], 3),                              # how long it took on this machine
            "receipts_issued": result["receipts_issued"],                        # payouts triggered by this block
        }), 201                                                                  # 201 means "created"

    @app.route("/payments", methods=["POST"])      # POST /payments runs the whole flow
    def create_payment():
        """Run on-ramp, transfer and off-ramp between two hosted wallets, mining after each step."""
        data = body()                                                            # the fields that were sent
        redeem = data.get("redeem", True)                                        # redeeming for the recipient is the default
        if not isinstance(redeem, bool):                                         # it must be true or false
            raise PlatformError(400, "'redeem' must be true or false")           # 400 means "bad request"
        payment = platform.pay(data.get("sender"), data.get("recipient"), data.get("amount"), redeem, data.get("bank_reference"))   # the whole flow
        return jsonify({"message": f"payment {payment['id']} completed", "payment": payment}), 201   # 201 means "created"

    @app.route("/payments/<payment_id>", methods=["GET"])   # GET /payments/<id> returns one payment
    def get_payment(payment_id):
        """Return the record of a payment made through POST /payments."""
        with platform.lock:                                                      # read while nothing is changing
            matches = [dict(payment) for payment in platform.payments if payment["id"] == payment_id]   # search by identifier
        if not matches:                                                          # no payment has that identifier
            raise PlatformError(404, "no payment with that identifier")          # 404 means "not found"
        return jsonify(matches[0]), 200                                          # the payment record



Appending to payment_api.py


### 7.4 Records, verification, explorer pages and errors

Refusals carry distinct status codes: 400 malformed, 401 bad signature, 403 not allowed, 409 replay, 422 insufficient funds.

In [33]:
%%writefile -a payment_api.py
    # -----------------------------------------------------------------------
    # Records
    # -----------------------------------------------------------------------
    @app.route("/reserve", methods=["GET"])        # GET /reserve reports on the reserve
    def get_reserve():
        """Return the reserve cash book set against the mZAR in circulation."""
        return jsonify(platform.reserve_report()), 200                           # the proof-of-reserves report

    @app.route("/receipts", methods=["GET"])       # GET /receipts lists every record
    def list_receipts():
        """Return every on-ramp conversion record and every off-ramp receipt."""
        view = platform.snapshot()                                               # one consistent reading
        return jsonify({"conversions": view["conversions"], "receipts": view["receipts"]}), 200   # the two lists

    @app.route("/receipts/<record_id>", methods=["GET"])   # GET /receipts/<id> returns one record
    def get_receipt(record_id):
        """Return one conversion record or receipt, found by its identifier or its transaction identifier."""
        record = platform.find_record(record_id)                                 # search both lists
        if record is None:                                                       # nothing matched
            raise PlatformError(404, "no conversion record or receipt with that identifier")   # 404 means "not found"
        return jsonify(record), 200                                              # the record

    # -----------------------------------------------------------------------
    # Verification
    # -----------------------------------------------------------------------
    @app.route("/chain", methods=["GET"])          # GET /chain returns every block
    def get_chain():
        """Return the full chain with the parameters needed to check it."""
        view = platform.snapshot()                               # one consistent reading
        return jsonify({                                         # reply in JSON
            "asset": ASSET,                                      # the stablecoin
            "native_asset": NATIVE_ASSET,                        # the mining-reward coin
            "difficulty": view["difficulty"],                    # the number of leading zeros every hash must show
            "genesis_hash": view["genesis_hash"],                # the fingerprint that identifies this network
            "reserve_agent": view["reserve_agent"],              # the only address that may mint
            "length": len(view["chain"]),                        # how many blocks, including genesis
            "chain": view["chain"],                              # the blocks, oldest first
        }), 200                                                  # 200 means "OK"

    @app.route("/chain/verify", methods=["GET"])   # GET /chain/verify audits the chain
    def verify_chain():
        """Replay the whole chain and report whether every rule holds."""
        view = platform.snapshot()                                               # one consistent reading
        return jsonify(audit_chain(view["chain"], view["genesis_hash"])), 200    # the same audit an outside verifier runs

    @app.route("/transactions/pending", methods=["GET"])   # GET /transactions/pending lists the pool
    def pending_transactions():
        """Return the transactions waiting to be mined."""
        pending = platform.snapshot()["pending"]                                 # a copy of the pool
        return jsonify({"count": len(pending), "transactions": pending}), 200    # how many, then the transactions

    @app.route("/transactions/<txid>", methods=["GET"])    # GET /transactions/<txid> looks one transaction up
    def get_transaction(txid):
        """Return one transaction with its block, its position and its confirmations."""
        with platform.lock:                                                      # read the ledger while nothing is changing it
            found = platform.ledger.find_transaction(txid)                       # search the chain, then the pool
        if found is None:                                                        # this node has never seen that identifier
            raise PlatformError(404, "no transaction with that identifier is known to this node")   # 404 means "not found"
        return jsonify(found), 200                                               # where the transaction is

    # -----------------------------------------------------------------------
    # Explorer pages
    # -----------------------------------------------------------------------
    @app.route("/explorer", methods=["GET"])       # GET /explorer returns a web page instead of JSON
    def explorer_page():
        """Return the overview as an HTML (HyperText Markup Language) page."""
        return Response(render_html(explorer_view()), mimetype="text/html")      # label the reply as a web page

    @app.route("/explorer/tx/<txid>", methods=["GET"])   # GET /explorer/tx/<txid> returns one payment record
    def explorer_payment_page(txid):
        """Return the payment record of one transaction as an HTML page."""
        return Response(render_payment_html(explorer_view(), txid), mimetype="text/html")   # label the reply as a web page

    # -----------------------------------------------------------------------
    # Errors
    # -----------------------------------------------------------------------
    @app.errorhandler(PlatformError)               # runs whenever the platform refuses a request
    def refused(error):
        """Answer a refused request with the platform's reason and status code."""
        return jsonify({"error": error.message}), error.status                   # the reason, with the status the platform chose

    @app.errorhandler(404)                         # runs when no endpoint matches the web address
    def not_found(_error):
        """Answer unknown addresses in JSON, like every other reply."""
        return jsonify({"error": "no such endpoint; GET / lists the endpoints"}), 404   # 404 means "not found"

    @app.errorhandler(405)                         # runs when the address exists but the method is wrong
    def wrong_method(_error):
        """Answer a wrong method (for example GET /mine) in JSON, like every other reply."""
        return jsonify({"error": "this endpoint does not accept that method; GET / lists the endpoints"}), 405   # 405 means "method not allowed"

    return app                                     # hand back the finished application



Appending to payment_api.py


### 7.5 Running a node in the background

In [34]:
%%writefile -a payment_api.py
# ===========================================================================
# RUNNING A NODE IN THE BACKGROUND (USED BY THE NOTEBOOK AND THE DEMONSTRATION SCRIPT)
# ===========================================================================

RUNNING = globals().get("RUNNING", {})   # background nodes by port; kept if this file is reloaded, so none are lost track of


def port_in_use(port: int) -> bool:
    """Return True if something on this computer is already listening on the port."""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as probe:   # open a temporary network socket
        probe.settimeout(0.5)                                          # give up after half a second
        return probe.connect_ex((HOST, port)) == 0                     # 0 means the connection succeeded, so the port is taken


def first_free_port(start: int = DEFAULT_PORT, tries: int = 20) -> int:
    """Return the first port from start onwards that this program already runs a node on, or that nothing is using."""
    for port in range(start, start + tries):                           # try the preferred port first, then the ones after it
        if port in RUNNING or not port_in_use(port):                   # our own node can be replaced, and an unused port is free
            return port                                                # this port can be used
    raise RuntimeError(f"no free port between {start} and {start + tries - 1}")   # every port tried was taken by another program


class BackgroundNode:
    """One node running inside this program on a background thread."""

    def __init__(self, port: int, **options):
        """Create the platform and start serving it straight away."""
        self.port = port                                             # the port this node listens on
        self.url = f"http://{HOST}:{port}"                           # the base web address of the node
        self.platform = RemittancePlatform(**options)                # the platform behind the API
        self.app = create_app(self.platform, port)                   # the Flask application
        self.server = make_server(HOST, port, self.app, threaded=True)   # a web server that handles each request on its own thread
        self.thread = threading.Thread(target=self.server.serve_forever, daemon=True)   # daemon=True lets the program exit without waiting
        self.thread.start()                                          # begin answering requests

    def stop(self):
        """Stop serving and release the port."""
        self.server.shutdown()             # ask the serving loop to finish
        self.server.server_close()         # close the listening socket so the port is free again
        self.thread.join()                 # wait for the background thread to end


def start_background_node(port: int = DEFAULT_PORT, **options) -> BackgroundNode:
    """Start a fresh node on the port, replacing any background node already running there."""
    if port in RUNNING:                                              # a node from an earlier run is still on this port
        RUNNING.pop(port).stop()                                     # stop it so the new node starts from a clean ledger
    if port_in_use(port):                                            # some other program holds the port
        raise RuntimeError(f"port {port} is already in use; close the program using it or choose another port")   # explain how to fix it
    logging.getLogger("werkzeug").setLevel(logging.ERROR)            # keep the web server's request log out of the output
    RUNNING[port] = BackgroundNode(port, **options)                  # start the node and remember it
    return RUNNING[port]                                             # hand it back to the caller


def stop_background_nodes():
    """Stop every background node this program started."""
    for port in list(RUNNING):             # list() copies the ports, because the dictionary shrinks as we go
        RUNNING.pop(port).stop()           # stop each node and forget it

# ===========================================================================
# RUNNING ONE NODE FROM THE COMMAND LINE
# ===========================================================================

def main():
    """Read the options from the command line and serve one node until Ctrl+C is pressed."""
    parser = argparse.ArgumentParser(description=f"Run one {ASSET} remittance node on this computer.")   # describes the program for --help
    parser.add_argument("--port", type=int, default=DEFAULT_PORT, help="port to listen on (default 5000)")   # where to listen
    parser.add_argument("--difficulty", type=int, default=DEMO_DIFFICULTY, help=f"leading zeros required of each block hash (default {DEMO_DIFFICULTY})")   # the mining rule
    parser.add_argument("--seed", help="derive repeatable keys from this text (demonstrations only); omit for random keys")   # repeatable keys
    parser.add_argument("--wall-clock", action="store_true", help="stamp records with the real time instead of the repeatable logical clock")   # which clock
    arguments = parser.parse_args()                                  # read what was typed
    if arguments.difficulty < MIN_DIFFICULTY:                        # the network's minimum from Assignment 7
        raise SystemExit(f"The difficulty must be at least {MIN_DIFFICULTY}.")   # stop with a clear message
    if port_in_use(arguments.port):                                  # refuse to start two nodes on one port
        raise SystemExit(f"Port {arguments.port} is already in use. Choose another with --port.")   # stop with a clear message
    clock = WallClock() if arguments.wall_clock else None            # None selects the repeatable logical clock
    platform = RemittancePlatform(difficulty=arguments.difficulty, seed=arguments.seed, clock=clock)   # build the platform
    app = create_app(platform, arguments.port)                       # wrap it in the API
    print(f"Node listening on http://{HOST}:{arguments.port}  (explorer at http://{HOST}:{arguments.port}/explorer)")   # where to find it
    print(f"Reserve agent {platform.reserve_agent.address}, genesis hash {platform.ledger.genesis_hash}")   # the two facts that identify the network
    app.run(host=HOST, port=arguments.port, threaded=True)           # serve on the loopback address only


if __name__ == "__main__":      # True only when this file is run directly, not when it is imported
    main()                      # start the node


Appending to payment_api.py


# Part 2: The demonstration

Steps i to vi of the brief, driven through the API exactly as an outside client would. Each request prints one line: the method, the endpoint and the status code.

## 8. Start the node

In [35]:
for name in ("simple_blockchain", "stablecoin", "remittance", "verify", "explorer", "payment_api"):   # the six files written above, lowest layer first
    importlib.reload(importlib.import_module(name))                         # load each file, and reload it in case it was written again

from explorer import fetch_snapshot, render_html, render_payment_html       # the explorer's reader and its web pages
from payment_api import first_free_port, start_background_node, stop_background_nodes   # find a port, then start and stop a node in this notebook
from simple_blockchain import Transaction, Wallet                           # used by the Sender, who signs outside the node
from stablecoin import ASSET                                                # the name of the stablecoin, "mZAR"
from verify import audit_chain, call_node, check_receipt, locate, render_audit, render_inclusion   # the outside party's checks

NODE_SEED = "blch9x2-capstone-demo"                                         # the fixed seed the node derives its demonstration keys from
CLIENT_SEED = "blch9x2-capstone-demo-client"                                # a separate seed for keys held outside the node

PORT = first_free_port()                                                    # 5000, or the next free port after it
NODE = f"http://127.0.0.1:{PORT}"                                           # the address of the node
start_background_node(PORT, seed=NODE_SEED)                                 # a fresh node, replacing any node left from an earlier run
print("Node started at", NODE)                                              # confirm where it is listening


def call(method, path, body=None):
    """Send one request to the node, print a one-line summary, and return the reply."""
    status, reply = call_node(method, NODE, path, body)                     # the request itself
    print(f"{method} {path}  ->  HTTP {status}")                            # what was asked, and the status code that came back
    return reply                                                            # the decoded reply, for the lines that follow


def table(headers, rows):
    """Print rows as a simple table with aligned columns."""
    text = [[str(cell) for cell in row] for row in [headers] + rows]        # every cell as text, headings first
    widths = [max(len(row[column]) for row in text) for column in range(len(headers))]   # each column as wide as its longest cell
    for row in text:                                                        # the headings, then each row
        print("  " + "  ".join(cell.ljust(width) for cell, width in zip(row, widths)))   # pad each cell to its column width


def position():
    """Print each wallet's confirmed mZAR, then the supply against the reserve."""
    print()                                                                 # a blank line before the table
    rows = []                                                               # one row per wallet
    for name, address in PEOPLE.items():                                    # each wallet in turn
        account = call_node("GET", NODE, f"/accounts/{address}")[1]         # read its account through the API
        rows.append([name, f"{account['confirmed']:,}"])                    # its name and confirmed balance
    table(["Wallet", f"Confirmed {ASSET}"], rows)                           # the balances
    reserve = call_node("GET", NODE, "/reserve")[1]                         # the reserve report
    print(f"  Supply {reserve['supply']:,} {ASSET}  |  reserve {reserve['reserve_balance']:,} ZAR  |  fully backed: {reserve['fully_backed']}")   # the reserve rule


def mined(reply):
    """Print the facts of a newly mined block."""
    block = reply["block"]                                                  # the block in the reply
    print(f"  Block {block['index']}: {len(block['transactions'])} records, nonce {block['nonce']:,} found after {reply['attempts']:,} hashes in {reply['seconds']} s")   # the work done
    print(f"  hash          {block['hash']}")                                # starts with four zeros
    print(f"  previous hash {block['previous_hash']}")                       # the link to the block before


def sign_locally(wallet, recipient, amount, kind="transfer"):
    """Build and sign a stablecoin transaction with a key the node never sees."""
    chain = call_node("GET", NODE, "/chain")[1]["chain"]                    # the chain is public, so anyone can read it
    account = call_node("GET", NODE, f"/accounts/{wallet.address}")[1]      # ask the node which sequence number comes next
    tx = Transaction(                                                       # the unsigned transaction
        sender=wallet.address,                                              # the signer's own address
        recipient=recipient,                                                # who is being paid
        amount=amount,                                                      # how much
        timestamp=chain[-1]["timestamp"] + 30,                              # a time just after the newest block
        kind=kind,                                                          # normally a transfer
        asset=ASSET,                                                        # always mZAR
        sequence=account["next_sequence"],                                  # the signer's next sequence number
        public_key=wallet.public_key_hex,                                   # so anyone can check the signature
    )
    wallet.sign_transaction(tx)                                             # the private key signs every field above, on this side
    return tx.to_dict()                                                     # only the signed transaction is sent to the node

Node started at http://127.0.0.1:5000


## 9. Step i: create the wallets

The Sender makes a key pair on her own device and registers only the address. The Recipient asks the platform to make and hold a key. The reserve agent and the validator belong to the operator.

In [36]:
status = call("GET", "/")                                                   # the node at the start: genesis only
sender = Wallet.from_seed(f"{CLIENT_SEED}/Sender", "Sender")                # the Sender's key pair, made outside the node
call("POST", "/wallets", {"name": "Sender", "address": sender.address})     # register only the address: self-custody
RECIPIENT = call("POST", "/wallets", {"name": "Recipient"})["wallet"]["address"]   # the platform makes and keeps this key: hosted
wallets = call("GET", "/wallets")["wallets"]                                # every wallet the platform knows
print()                                                                     # a blank line before the table
rows = [[w["name"], w["custody"], w["address"]] for w in wallets]          # name, custody and address; never a private key
table(["Name", "Custody", "Address"], rows)                                 # one row per wallet
print(f"\n  Chain length {status['chain_length']}; block 0 names the reserve agent")   # only the genesis block exists
AGENT = status["reserve_agent"]                                             # the reserve agent's address
PEOPLE = {"Sender": sender.address, "Recipient": RECIPIENT, "Reserve agent": AGENT}   # balances shown below

GET /  ->  HTTP 200
POST /wallets  ->  HTTP 201
POST /wallets  ->  HTTP 201
GET /wallets  ->  HTTP 200

  Name           Custody   Address                                 
  Reserve agent  operator  0d8cb096595dcf1ed5b2ad861890b98f3cd96176
  Validator      operator  642908387887e8e30930847b7afc2b0d8e660304
  Sender         self      1f20731db9e34b92d4609b65ca7a029a7ebdde3f
  Recipient      hosted    e2637520d3ed6a039b23ff672bfbefaabcb72174

  Chain length 1; block 0 names the reserve agent


POST /wallets  ->  HTTP 201
POST /wallets  ->  HTTP 201
GET /wallets  ->  HTTP 200

  Name           Custody   Address                                 
  Reserve agent  operator  0d8cb096595dcf1ed5b2ad861890b98f3cd96176
  Validator      operator  642908387887e8e30930847b7afc2b0d8e660304
  Sender         self      1f20731db9e34b92d4609b65ca7a029a7ebdde3f
  Recipient      hosted    e2637520d3ed6a039b23ff672bfbefaabcb72174

  Chain length 1; block 0 names the reserve agent


## 10. Step ii: on-ramp, 1 000 rand in and 1 000 mZAR out

The reserve records the rand first, then the reserve agent signs the mint.

In [37]:
deposit = {"address": sender.address, "amount": 1000, "bank_reference": "EFT-20261013-001"}   # 1 000 rand
reply = call("POST", "/onramp", deposit)                                    # the Sender deposits the rand
c = reply["conversion"]                                                     # the record of this conversion
print(f"  {c['id']}: {c['deposited']:,} ZAR in, {c['minted']:,} {ASSET} minted, status {c['status']}")   # what happened
print(f"  reserve holds {reply['reserve_balance']:,} ZAR; mint txid {c['txid']}")   # money in first

POST /onramp  ->  HTTP 201
  DEP-0001: 1,000 ZAR in, 1,000 mZAR minted, status pending
  reserve holds 1,000 ZAR; mint txid 6251b477c8ad865ca220e4e727e99663647290bf8a04f3ca4e9817ea0ddd9c72


In [38]:
mined(call("POST", "/mine"))   # block 1 carries the reward and the mint
position()                     # the Sender now holds 1 000 mZAR

POST /mine  ->  HTTP 201
  Block 1: 2 records, nonce 63,361 found after 63,362 hashes in 1.492 s
  hash          0000d7e464bf3761ee07a0750b742dda484148277bb7c65f582782c2a4cff40c
  previous hash 85c4405998fb7eab0f33418a16412e86cec9d2201ca637ac0eb215546a9aa60c

  Wallet         Confirmed mZAR
  Sender         1,000         
  Recipient      0             
  Reserve agent  0             
  Supply 1,000 mZAR  |  reserve 1,000 ZAR  |  fully backed: True


## 11. Step iii: the Sender signs 750 mZAR to the Recipient

The transaction is built and signed on the Sender's side. Only the signed result reaches the node.

In [39]:
signed_transfer = sign_locally(sender, RECIPIENT, 750)                      # signed with a key the node has never seen
print(json.dumps(signed_transfer, indent=2))                                # the signed transaction exactly as it is sent
TRANSFER_TXID = call("POST", "/transactions", signed_transfer)["txid"]      # submit it and keep its identifier
pending = call_node("GET", NODE, f"/transactions/{TRANSFER_TXID}")[1]       # look it up before mining
print(f"  txid {TRANSFER_TXID}: {pending['status']}, {pending['confirmations']} confirmations")   # waiting in the pool

{
  "sender": "1f20731db9e34b92d4609b65ca7a029a7ebdde3f",
  "recipient": "e2637520d3ed6a039b23ff672bfbefaabcb72174",
  "amount": 750,
  "timestamp": 1790000150,
  "signature": "30450220767059fd5bfd6f6d3753bf2d7b4a32098d86200f29170a64e0cdd58f5bbef7d2022100afb273b5c2e2d091db83e621511706c716c2edb747b48abb1a5c0a05cadace52",
  "kind": "transfer",
  "asset": "mZAR",
  "sequence": 0,
  "public_key": "016b3da12aca62fe6f421048212bc4697c6a2096c89d752813891c4166beede4bcbef27855f53d6c8a6e77b12d6eb9ca01a3d1f5ee7ba0cfb9f9e7fdfeaff87f"
}
POST /transactions  ->  HTTP 201
  txid 3c61f3cd84569501ace3677890e846d41cbcf6cab0ea47b1f9537cfbf06d590f: pending, 0 confirmations


In [40]:
mined(call("POST", "/mine"))                                                # block 2 carries the signed transfer
found = call("GET", f"/transactions/{TRANSFER_TXID}")                       # the transfer after mining
print(f"  {found['status']} in block {found['block_index']}, {found['confirmations']} confirmation")   # now part of the chain
position()                                                                  # 250 stays with the Sender, 750 is with the Recipient

POST /mine  ->  HTTP 201
  Block 2: 2 records, nonce 14,622 found after 14,623 hashes in 0.346 s
  hash          00000bf91e0edf80f73a2869df04bb388eec46c6046958cea036dc0491c4cf9c
  previous hash 0000d7e464bf3761ee07a0750b742dda484148277bb7c65f582782c2a4cff40c
GET /transactions/3c61f3cd84569501ace3677890e846d41cbcf6cab0ea47b1f9537cfbf06d590f  ->  HTTP 200
  confirmed in block 2, 1 confirmation

  Wallet         Confirmed mZAR
  Sender         250           
  Recipient      750           
  Reserve agent  0             
  Supply 1,000 mZAR  |  reserve 1,000 ZAR  |  fully backed: True


  Wallet         Confirmed mZAR
  Sender         250           
  Recipient      750           
  Reserve agent  0             
  Supply 1,000 mZAR  |  reserve 1,000 ZAR  |  fully backed: True


## 12. Step iv: off-ramp, 750 mZAR in and 750 loti out

The platform signs a burn with the Recipient's hosted key. Nothing is paid until the burn is mined.

In [41]:
burn = call("POST", "/offramp", {"address": RECIPIENT, "amount": 750})      # HTTP 202: accepted, not yet completed
print(f"  burn txid {burn['txid']}: {burn['status']}")                      # waiting for a block
reserve = call_node("GET", NODE, "/reserve")[1]                             # the reserve before mining
print(f"  reserve still holds {reserve['reserve_balance']:,} ZAR")          # nothing leaves before the burn is mined

POST /offramp  ->  HTTP 202
  burn txid ecad44c787736a5f91563538b53331b1b900fa76805a7361eee5f58d8ae517ad: awaiting confirmation
  reserve still holds 1,000 ZAR


In [42]:
reply = call("POST", "/mine")                                               # block 3 carries the burn, which triggers the payout
mined(reply)                                                                # the block's facts
receipt = reply["receipts_issued"][0]                                       # the receipt issued when the burn was confirmed
r = receipt                                                                 # a short name for the lines below
print(f"\n  Receipt {r['id']}: {r['burned']:,} {ASSET} burned, {r['paid_out']:,} LSL paid, status {r['status']}")   # the outcome
print(f"  burn {r['txid']} in block {r['block_index']}")                   # the link to the chain
print(f"  signed by the reserve agent {r['issuer']}")                       # the signer, as named in block 0
position()                                                                  # 250 rand in reserve behind the Sender's 250 mZAR

POST /mine  ->  HTTP 201
  Block 3: 2 records, nonce 55,685 found after 55,686 hashes in 1.347 s
  hash          0000ff511afc83cea88fe4df8755e5c9aa1f48ac0f519d90d2f858a6f2c11d7d
  previous hash 00000bf91e0edf80f73a2869df04bb388eec46c6046958cea036dc0491c4cf9c

  Receipt RCPT-0001: 750 mZAR burned, 750 LSL paid, status paid
  burn ecad44c787736a5f91563538b53331b1b900fa76805a7361eee5f58d8ae517ad in block 3
  signed by the reserve agent 0d8cb096595dcf1ed5b2ad861890b98f3cd96176

  Wallet         Confirmed mZAR
  Sender         250           
  Recipient      0             
  Reserve agent  0             
  Supply 250 mZAR  |  reserve 250 ZAR  |  fully backed: True


## 13. Step v: verification by an outside party

### 13.1 The node's own audit, then an independent one

The independent audit takes only the blocks from the node and recomputes everything.

In [43]:
node_audit = call("GET", "/chain/verify")                                   # the node's verdict on its own chain
print(f"  node says valid: {node_audit['valid']}\n")                        # what the node reports
downloaded = call_node("GET", NODE, "/chain")[1]                            # the only data taken from the node: the blocks
chain = downloaded["chain"]                                                 # the list of blocks
audit = audit_chain(chain, downloaded["genesis_hash"])                      # replayed on this side, from genesis
print(render_audit(audit))                                                  # the independent verdict

GET /chain/verify  ->  HTTP 200
  node says valid: True

CHAIN AUDIT (every check recomputed locally from the blocks the node supplied)
  block 0  85c4405998fb7eab0f33418a16412e86cec9d2201ca637ac0eb215546a9aa60c
           1 record(s), 0 signature(s) checked: charter names reserve agent 0d8cb096595dcf1ed5b2ad861890b98f3cd96176
  block 1  0000d7e464bf3761ee07a0750b742dda484148277bb7c65f582782c2a4cff40c
           2 record(s), 1 signature(s) checked: hash, link, proof of work, reward, signatures, sequences and balances all hold
  block 2  00000bf91e0edf80f73a2869df04bb388eec46c6046958cea036dc0491c4cf9c
           2 record(s), 1 signature(s) checked: hash, link, proof of work, reward, signatures, sequences and balances all hold
  block 3  0000ff511afc83cea88fe4df8755e5c9aa1f48ac0f519d90d2f858a6f2c11d7d
           2 record(s), 1 signature(s) checked: hash, link, proof of work, reward, signatures, sequences and balances all hold
  RESULT: VALID. 4 blocks, 3 signatures, 1,000 mZAR minted, 75

### 13.2 Transaction lookup and the receipt

The transfer is located in block 2, and the receipt is checked against the chain and the reserve agent's signature.

In [44]:
found = locate(chain, TRANSFER_TXID)                                        # find the transfer in the downloaded blocks
print(render_inclusion(found, TRANSFER_TXID))                               # its block, signature and confirmations
print()                                                                     # a blank line before the receipt
tests_passed = check_receipt(receipt, chain, audit["reserve_agent"])        # five independent tests of the receipt
table(["Receipt test", "Result"], [[t, "yes" if ok else "NO"] for t, ok in tests_passed.items()])   # one row per test

TRANSACTION 3c61f3cd84569501ace3677890e846d41cbcf6cab0ea47b1f9537cfbf06d590f
  Contents      : transfer of 750 from 1f20731db9e34b92d4609b65ca7a029a7ebdde3f to e2637520d3ed6a039b23ff672bfbefaabcb72174
  Signature     : valid
  Included in   : block 2, position 1
  Block hash    : 00000bf91e0edf80f73a2869df04bb388eec46c6046958cea036dc0491c4cf9c
  Recomputed    : 00000bf91e0edf80f73a2869df04bb388eec46c6046958cea036dc0491c4cf9c (matches)
  Confirmations : 2

  Receipt test             Result
  signed_by_reserve_agent  yes   
  burn_is_on_chain         yes   
  block_matches            yes   
  customer_matches         yes   
  amount_matches           yes   


### 13.3 Tamper evidence

In a private copy of the chain, the confirmed 750 is changed to 7 500. The audit rejects the copy at block 2.

In [45]:
forged = copy.deepcopy(chain)                                               # a private copy, so the node itself is untouched
forged[2]["transactions"][1]["amount"] = 7500                               # change the confirmed 750 into 7 500
print(render_audit(audit_chain(forged, downloaded["genesis_hash"])))        # audit the edited copy

CHAIN AUDIT (every check recomputed locally from the blocks the node supplied)
  RESULT: INVALID. Block 2: self-hash pillar failed (stored=00000bf91e0e..., recomputed=a7e62ab959e7...).


## 14. Step v: the explorer

The two pages the node serves at `/explorer/tx/<transaction id>` and `/explorer`, shown here.

In [46]:
snapshot = fetch_snapshot(NODE)                                 # read the node through its public endpoints
display(HTML(render_payment_html(snapshot, TRANSFER_TXID)))     # the payment record: full transaction and block identifiers

Item,Value
Transaction ID,3c61f3cd84569501ace3677890e846d41cbcf6cab0ea47b1f9537cfbf06d590f
Kind,transfer
Amount,750 mZAR
From,Sender 1f20731db9e34b92d4609b65ca7a029a7ebdde3f
To,Recipient e2637520d3ed6a039b23ff672bfbefaabcb72174
Sequence,0
Reference,-
Created (UTC),2026-09-21 14:15
Signature,valid (checked against the public key carried in the transaction)
Status,"confirmed, 2 confirmation(s)"


In [47]:
display(HTML(render_html(snapshot)))   # the overview: blocks, transactions, balances, receipts and the reserve

Item,Value
Node,http://127.0.0.1:5000
Asset,mZAR. 1 mZAR = 1 ZAR held in reserve
Reserve agent,0d8cb096595dcf1ed5b2ad861890b98f3cd96176
Genesis hash,85c4405998fb7eab0f33418a16412e86cec9d2201ca637ac0eb215546a9aa60c
Chain height,3 (4 blocks including genesis)
Difficulty,4 leading zeros
Chain check,"valid: every hash, link, proof of work, signature and balance was recomputed by this explorer"
Supply,"250 mZAR in circulation (1,000 minted, 750 burned)"
Reserve,250 ZAR held. Fully backed: yes
Pending,0 transaction(s) waiting to be mined


## 15. Step vi: invalid requests are refused

An attacker with her own key and no registration tries six things. The chain is public, so she starts by copying the Sender's confirmed transfer from it.

In [48]:
mallory = Wallet.from_seed(f"{CLIENT_SEED}/Mallory", "Mallory")             # the attacker's own key pair
confirmed_transfer = locate(chain, TRANSFER_TXID)["transaction"]            # the Sender's transfer, copied from the public chain
tip_before = chain[-1]["hash"]                                              # the newest block hash before the attempts
forged_transfer = sign_locally(mallory, mallory.address, 250)               # a transaction the attacker can sign validly
forged_transfer["sender"], forged_transfer["sequence"] = sender.address, 1  # then relabelled as coming from the Sender

redirected = dict(confirmed_transfer, recipient=mallory.address)            # same signature, different recipient
too_much = sign_locally(sender, RECIPIENT, 500)                             # genuine, but more than the Sender holds
self_mint = sign_locally(mallory, mallory.address, 1_000_000, kind="mint")  # validly signed, but not by the reserve agent
unregistered = {"address": mallory.address, "amount": 1000}                 # an address that was never registered

attempts = [                                                                # a label, an endpoint and the body sent
    ("(a) copied transfer, recipient changed", "/transactions", redirected),         # bad signature
    ("(b) attacker's key on the Sender's account", "/transactions", forged_transfer),   # wrong key
    ("(c) Sender holds 250, signs 500", "/transactions", too_much),                   # insufficient balance
    ("(d) confirmed transfer sent again", "/transactions", confirmed_transfer),       # replay
    ("(e) attacker mints 1 000 000 to herself", "/transactions", self_mint),          # forged mint
    ("(f) on-ramp for an unregistered address", "/onramp", unregistered),             # not a registered customer
]
rows = []                                                                   # one row per attempt
for label, path, body in attempts:                                          # send each attempt in turn
    status, reply = call_node("POST", NODE, path, body)                     # the request itself
    rows.append([label, status, reply["error"]])                            # what came back
table(["Attempt", "HTTP", "Reply from the node"], rows)                     # the six refusals

pool = call_node("GET", NODE, "/transactions/pending")[1]                   # the pending pool after the attempts
tip_after = call_node("GET", NODE, "/chain")[1]["chain"][-1]["hash"]        # the newest block hash after the attempts
reserve = call_node("GET", NODE, "/reserve")[1]                             # the reserve after the attempts
print(f"\n  pending pool: {pool['count']}  |  newest block unchanged: {tip_after == tip_before}")   # nothing entered
print(f"  reserve {reserve['reserve_balance']} ZAR, supply {reserve['supply']} {ASSET}")   # still equal

  Attempt                                     HTTP  Reply from the node                                                 
  (a) copied transfer, recipient changed      401   the signature does not match the transaction's contents             
  (b) attacker's key on the Sender's account  401   the public key does not belong to the sender's address              
  (c) Sender holds 250, signs 500             422   the sender holds 250 mZAR but the transaction needs 500             
  (d) confirmed transfer sent again           409   this transaction is already on the ledger                           
  (e) attacker mints 1 000 000 to herself     403   only the reserve agent named in the genesis block may mint          
  (f) on-ramp for an unregistered address     403   that address is not a registered customer; register the wallet first

  pending pool: 0  |  newest block unchanged: True
  reserve 250 ZAR, supply 250 mZAR


## 16. The whole payment in one API call

A second customer with a hosted wallet sends 400. `POST /payments` runs the on-ramp, the transfer and the off-ramp, mining a block after each.

In [49]:
SECOND = call("POST", "/wallets", {"name": "Second sender"})["wallet"]["address"]   # a hosted wallet
order = {"sender": SECOND, "recipient": RECIPIENT, "amount": 400, "bank_reference": "EFT-20261013-002"}   # the payment instruction
payment = call("POST", "/payments", order)["payment"]                       # the whole flow in one call
p = payment                                                                 # a short name for the lines below
print(f"\n  {p['id']}: {p['amount']} {p['status']}, deposit {p['conversion_id']}, receipt {p['receipt_id']}")   # the outcome
rows = [[step["step"], step["block_index"], step["txid"]] for step in p["steps"]]   # one row per step
table(["Step", "Block", "Transaction ID"], rows)                            # the evidence for each step

POST /wallets  ->  HTTP 201
POST /payments  ->  HTTP 201

  PAY-0001: 400 paid out, deposit DEP-0002, receipt RCPT-0002
  Step             Block  Transaction ID                                                  
  on-ramp (mint)   4      4e78bd8d5088c8e64cb3afbacba31f69446b4e73907ee8cc66c2b5ae74efef0c
  transfer         5      702a364773ca01a98efdbe685110260943e3e0c493940a728cca97a2be895057
  off-ramp (burn)  6      d5a967662009e42f27076bde8ccb76073c212f562f06950909d35cb9ffe637aa


POST /payments  ->  HTTP 201

  PAY-0001: 400 paid out, deposit DEP-0002, receipt RCPT-0002
  Step             Block  Transaction ID                                                  
  on-ramp (mint)   4      4e78bd8d5088c8e64cb3afbacba31f69446b4e73907ee8cc66c2b5ae74efef0c
  transfer         5      702a364773ca01a98efdbe685110260943e3e0c493940a728cca97a2be895057
  off-ramp (burn)  6      d5a967662009e42f27076bde8ccb76073c212f562f06950909d35cb9ffe637aa


# Part 3: Checks

## 17. Checks on the run

Each check reads the node through the API.

In [50]:
final = call_node("GET", NODE, "/chain")[1]                                 # the chain at the end
blocks = final["chain"]                                                     # the list of blocks
final_audit = audit_chain(blocks, final["genesis_hash"])                    # audited on this side
reserve = call_node("GET", NODE, "/reserve")[1]                             # the reserve report
held = {name: call_node("GET", NODE, f"/accounts/{address}")[1]["confirmed"] for name, address in PEOPLE.items()}   # the confirmed balances
transfer_now = call_node("GET", NODE, f"/transactions/{TRANSFER_TXID}")[1]  # the Sender's transfer, as the node reports it now
receipts = call_node("GET", NODE, "/receipts")[1]                           # every conversion record and receipt

checks = [                                                                  # each check is a description and a True or False
    ("the chain holds 7 blocks", final["length"] == 7),                     # genesis plus six mined blocks
    ("the outside audit finds the chain valid", final_audit["valid"]),      # every rule holds from genesis
    ("6 signatures were verified", final_audit["signatures_checked"] == 6), # two mints, two transfers and two burns
    ("every mined block hash starts with four zeros", all(block["hash"].startswith("0000") for block in blocks[1:])),   # proof of work
    ("every block points at the block before it", all(later["previous_hash"] == earlier["hash"] for earlier, later in zip(blocks, blocks[1:]))),   # the links
    ("the Sender's transfer is confirmed in block 2", transfer_now["status"] == "confirmed" and transfer_now["block_index"] == 2),   # where it was mined
    ("the Sender's transfer has 5 confirmations", transfer_now["confirmations"] == 5),   # blocks 2 to 6
    ("the Sender holds 250 and the Recipient 0", held["Sender"] == 250 and held["Recipient"] == 0),   # after both redemptions
    ("the reserve agent holds no mZAR", held["Reserve agent"] == 0),        # burned mZAR is destroyed, not kept
    ("1 400 minted and 1 150 burned", (reserve["minted"], reserve["burned"]) == (1400, 1150)),   # both payments together
    ("250 mZAR in circulation, 250 rand in reserve", (reserve["supply"], reserve["reserve_balance"]) == (250, 250)),   # the reserve rule
    ("two deposits confirmed and two receipts paid", [item["status"] for item in receipts["conversions"] + receipts["receipts"]] == ["confirmed", "confirmed", "paid", "paid"]),   # the records
    ("the pending pool is empty", call_node("GET", NODE, "/transactions/pending")[1]["count"] == 0),   # nothing is waiting
]
for description, passed in checks:                                          # list every check
    print(f"[{'PASS' if passed else 'FAIL'}] {description}")                # with its outcome
print()                                                                     # a blank line before the count
print(sum(passed for _, passed in checks), "passed,", sum(not passed for _, passed in checks), "failed")   # the final count

[PASS] the chain holds 7 blocks
[PASS] the outside audit finds the chain valid
[PASS] 6 signatures were verified
[PASS] every mined block hash starts with four zeros
[PASS] every block points at the block before it
[PASS] the Sender's transfer is confirmed in block 2
[PASS] the Sender's transfer has 5 confirmations
[PASS] the Sender holds 250 and the Recipient 0
[PASS] the reserve agent holds no mZAR
[PASS] 1 400 minted and 1 150 burned
[PASS] 250 mZAR in circulation, 250 rand in reserve
[PASS] two deposits confirmed and two receipts paid
[PASS] the pending pool is empty

13 passed, 0 failed


## 18. Rule tests: `tests.py`

The rules tested directly, without the web server, including cases the demonstration does not reach: a forged block that has been mined again, a second spend waiting in the pool, a redemption by an unregistered address, and a competing chain.

### 18.1 Helpers

In [51]:
%%writefile tests.py
"""
tests.py
Assignment 9: Financial Blockchain Capstone (the rule tests)
Blockchain module (BLCH9X2), Master of Financial Engineering, University of Johannesburg

WHAT THIS PROGRAM DOES
    Tests the rules directly, without the web server, including cases the demonstration
    does not reach: a forged block that has been mined again, a second spend waiting in
    the pool, a redemption by an unregistered address, and a competing chain.

        Group 1  the code carried forward from Assignments 1 to 7 still behaves as before
        Group 2  the rules for one stablecoin transaction
        Group 3  the rules for a whole chain, and the fork-choice rule
        Group 4  the platform: custody, the reserve and receipts

HOW TO RUN
    python tests.py          ends with a count such as "26 passed, 0 failed"

    The tests mine at difficulty 2, so the whole file runs in a second or two.
"""

import copy                 # copy.deepcopy makes an independent duplicate of a chain, so a test can damage it safely

from remittance import PlatformError, RemittancePlatform, receipt_signature_valid   # the platform, its error type and the receipt check
from simple_blockchain import (     # the chain core
    BLOCK_REWARD,                   # the miner's reward
    Transaction,                    # one transaction
    Wallet,                         # a key pair
    create_genesis_block,           # builds block 0
    make_coinbase,                  # builds a miner's reward
    mine_with_stats,                # the proof-of-work search
    tx_id,                          # the identifier of a transaction
)
from stablecoin import ASSET, Ledger, replay   # the asset name, the ledger and the full chain validation

TEST_DIFFICULTY = 2         # the lowest difficulty the network allows, so the tests run quickly
TESTS = []                  # every test function is added to this list by the decorator below

# ===========================================================================
# HELPERS
# ===========================================================================

def test(function):
    """Register a function as a test. Written above a function as @test."""
    TESTS.append(function)      # remember the function so the runner can call it
    return function             # hand the function back unchanged


def new_platform(**options) -> tuple:
    """Return a platform with repeatable keys and two hosted customers: (platform, sender address, recipient address)."""
    platform = RemittancePlatform(difficulty=TEST_DIFFICULTY, seed="tests", **options)   # repeatable keys, easy mining
    sender = platform.register_wallet("Sender")["address"]              # a hosted wallet for the paying customer
    recipient = platform.register_wallet("Recipient")["address"]        # a hosted wallet for the receiving customer
    return platform, sender, recipient                                  # everything a test needs to start


def signed(wallet: Wallet, ledger: Ledger, recipient: str, amount, kind: str = "transfer", sequence=None, timestamp: int = 1_790_009_999) -> dict:
    """Build and sign a stablecoin transaction outside the platform."""
    if sequence is None:                                                # normally the next sequence number is looked up
        sequence = ledger.account(wallet.address)["next_sequence"]      # the number the ledger expects
    tx = Transaction(sender=wallet.address, recipient=recipient, amount=amount, timestamp=timestamp,   # the four signed fields from Assignment 4
                     kind=kind, asset=ASSET, sequence=sequence, public_key=wallet.public_key_hex)     # and the capstone fields
    wallet.sign_transaction(tx)                                         # sign every field above
    return tx.to_dict()                                                 # the signed transaction as a dictionary


def funded_ledger() -> tuple:
    """Return (platform, ledger, Alice's wallet, Bob's address), where Alice keeps her own key and holds 100 mZAR."""
    platform, _, bob = new_platform()                                   # a platform with two hosted customers
    alice = Wallet.from_seed("tests/alice", "Alice")                    # a key the platform does not hold
    platform.register_wallet("Alice", alice.address)                    # registered by address only
    platform.on_ramp(alice.address, 100)                                # Alice deposits 100 rand
    platform.mine()                                                     # the mint is confirmed in block 1
    return platform, platform.ledger, alice, bob                        # everything a rule test needs


def mine_again(blocks: list, start: int):
    """Repair the links and proof of work of a chain from one block onwards, as a determined forger would."""
    for position in range(start, len(blocks)):                          # the edited block and every block after it
        blocks[position].previous_hash = blocks[position - 1].hash      # point at the (possibly new) hash of the parent
        mine_with_stats(blocks[position], blocks[position].difficulty)  # redo the proof of work


def refused(action, status: int) -> bool:
    """Return True if calling action() is refused by the platform with the expected status code."""
    try:                                    # the call is expected to raise a PlatformError
        action()                            # make the call
    except PlatformError as error:          # the platform refused it
        return error.status == status       # True only if it refused for the expected reason
    return False                            # the call went through, which the test did not expect



Overwriting tests.py


### 18.2 Group 1: the code carried forward from Assignments 1 to 7

In [52]:
%%writefile -a tests.py
# ===========================================================================
# GROUP 1: THE CODE CARRIED FORWARD FROM ASSIGNMENTS 1 TO 7
# ===========================================================================

@test
def assignment_2_block_hash_is_unchanged():
    """The capstone extensions did not change how a block from Assignment 2 is hashed."""
    genesis = create_genesis_block(timestamp=1_700_000_000)     # the block printed in the Assignment 2 notebook
    assert genesis.hash.startswith("6b5ab8051e")                # the same hash as it had then


@test
def five_field_transaction_is_unchanged():
    """A transaction that uses none of the new fields looks and signs exactly as it did in Assignments 3 and 4."""
    tx = Transaction("alice", "bob", 10, timestamp=1)           # the five-field form
    assert list(tx.to_dict()) == ["sender", "recipient", "amount", "timestamp", "signature"]   # no new field appears
    assert tx.payload_bytes() == b'{"amount":10,"recipient":"bob","sender":"alice","timestamp":1}'   # the signed bytes are the same


@test
def seeded_keys_and_signatures_are_repeatable():
    """The same seed gives the same key and signature every time, and a random wallet differs."""
    first, second = Wallet.from_seed("tests/same"), Wallet.from_seed("tests/same")   # two wallets from one seed
    assert first.address == second.address                      # the same key, so the same address
    assert first.sign(b"message") == second.sign(b"message")    # deterministic signing gives the same signature
    assert Wallet.create().address != first.address             # a random key is different


@test
def a_signature_covers_every_field():
    """Changing any signed field, including a capstone field, breaks the signature."""
    wallet = Wallet.from_seed("tests/signer")                   # the signer
    tx = Transaction(wallet.address, "b" * 40, 5, timestamp=1, kind="transfer", asset=ASSET, sequence=0, public_key=wallet.public_key_hex)   # a full transaction
    wallet.sign_transaction(tx)                                 # sign it
    assert Wallet.verify(tx.payload_bytes(), tx.signature, wallet.public_key_hex)   # the untouched transaction verifies
    for field, value in (("amount", 6), ("recipient", "c" * 40), ("kind", "burn"), ("sequence", 1)):   # one change at a time
        altered = Transaction.from_dict({**tx.to_dict(), field: value})             # the same transaction with one field changed
        assert not Wallet.verify(altered.payload_bytes(), altered.signature, wallet.public_key_hex)   # the signature no longer holds



Appending to tests.py


### 18.3 Group 2: the rules for one transaction

In [53]:
%%writefile -a tests.py
# ===========================================================================
# GROUP 2: THE RULES FOR ONE TRANSACTION
# ===========================================================================

@test
def a_valid_transfer_is_accepted_and_moves_value():
    """A correctly signed transfer is accepted, and after mining both balances change."""
    platform, ledger, alice, bob = funded_ledger()              # Alice holds 100 mZAR
    assert ledger.submit(signed(alice, ledger, bob, 40))[0] == "accepted"   # the transfer enters the pool
    platform.mine()                                             # it is mined into block 2
    assert ledger.account(alice.address)["confirmed"] == 60 and ledger.account(bob)["confirmed"] == 40   # 40 moved from Alice to Bob


@test
def only_the_reserve_agent_may_mint():
    """A mint signed by anyone other than the address in the genesis charter is refused."""
    _, ledger, alice, _ = funded_ledger()                       # Alice is an ordinary customer
    assert ledger.submit(signed(alice, ledger, alice.address, 1_000_000, kind="mint"))[0] == "unauthorised"   # she cannot create mZAR


@test
def nobody_can_spend_more_than_they_hold():
    """A transfer or burn larger than the balance is refused."""
    platform, ledger, alice, bob = funded_ledger()              # Alice holds 100 mZAR
    assert ledger.submit(signed(alice, ledger, bob, 101))[0] == "insufficient_funds"   # one unit too many
    assert ledger.submit(signed(alice, ledger, platform.reserve_agent.address, 101, kind="burn"))[0] == "insufficient_funds"   # the same for a burn


@test
def an_altered_transaction_is_refused():
    """A transaction changed after signing, or signed with somebody else's key, is refused."""
    _, ledger, alice, bob = funded_ledger()                     # Alice holds 100 mZAR
    mallory = Wallet.from_seed("tests/mallory")                 # an attacker with her own key
    genuine = signed(alice, ledger, bob, 10)                    # a transfer Alice really signed
    assert ledger.submit({**genuine, "amount": 100})[0] == "bad_signature"              # the amount was raised afterwards
    assert ledger.submit({**genuine, "recipient": mallory.address})[0] == "bad_signature"   # the recipient was swapped afterwards
    forged = {**signed(mallory, ledger, mallory.address, 10), "sender": alice.address}  # signed by Mallory, relabelled as Alice
    assert ledger.submit(forged)[0] == "bad_signature"          # Mallory's key does not belong to Alice's address


@test
def a_transaction_cannot_be_replayed():
    """The same signed transaction is refused a second time, both while pending and after it is mined."""
    platform, ledger, alice, bob = funded_ledger()              # Alice holds 100 mZAR
    transfer = signed(alice, ledger, bob, 10)                   # a genuine transfer
    assert ledger.submit(transfer)[0] == "accepted"             # accepted the first time
    assert ledger.submit(transfer)[0] == "replay"               # refused while it is still pending
    platform.mine()                                             # mine it
    assert ledger.submit(transfer)[0] == "replay"               # refused again once it is confirmed


@test
def sequence_numbers_must_be_used_in_order():
    """A transaction carrying a sequence number that is too low or too high is refused."""
    platform, ledger, alice, bob = funded_ledger()              # Alice has sent nothing, so her next number is 0
    assert ledger.submit(signed(alice, ledger, bob, 10, sequence=1))[0] == "replay"   # a gap: number 0 was skipped
    assert ledger.submit(signed(alice, ledger, bob, 10, sequence=0))[0] == "accepted" # the right number
    assert ledger.submit(signed(alice, ledger, bob, 20, sequence=0))[0] == "replay"   # number 0 used again for a different payment
    assert ledger.account(alice.address)["next_sequence"] == 1  # the pending transfer already counts


@test
def two_pending_spends_cannot_overdraw():
    """A second transfer is checked against what the first one, still pending, will leave behind."""
    _, ledger, alice, bob = funded_ledger()                     # Alice holds 100 mZAR
    assert ledger.submit(signed(alice, ledger, bob, 60))[0] == "accepted"             # 40 will remain
    assert ledger.submit(signed(alice, ledger, bob, 60))[0] == "insufficient_funds"   # so a second 60 is refused before any block is mined


@test
def malformed_transactions_are_refused():
    """Decimals, booleans, unknown fields, null fields and a transfer to oneself are all refused as malformed."""
    platform, ledger, alice, bob = funded_ledger()              # Alice holds 100 mZAR
    assert ledger.submit(signed(alice, ledger, bob, 2.5))[0] == "malformed"             # a decimal amount
    assert ledger.submit(signed(alice, ledger, bob, True))[0] == "malformed"            # a boolean, which Python would count as 1
    assert ledger.submit({**signed(alice, ledger, bob, 10), "note": "hello"})[0] == "malformed"   # a field outside the schema
    assert ledger.submit({**signed(alice, ledger, bob, 10), "reference": None})[0] == "malformed" # a field set to null
    assert ledger.submit(signed(alice, ledger, alice.address, 10))[0] == "malformed"    # paying yourself
    assert ledger.submit(signed(alice, ledger, bob, 10, kind="burn"))[0] == "unauthorised"   # a burn not addressed to the reserve agent
    assert ledger.submit("not a transaction")[0] == "malformed" # not even a JSON object
    assert ledger.pending == []                                 # nothing refused ever entered the pool



Appending to tests.py


### 18.4 Group 3: the rules for a whole chain

In [54]:
%%writefile -a tests.py
# ===========================================================================
# GROUP 3: THE RULES FOR A WHOLE CHAIN
# ===========================================================================

def payment_chain() -> tuple:
    """Return (platform, a copy of its chain) after one full payment: mint in block 1, transfer in block 2, burn in block 3."""
    platform, sender, recipient = new_platform()                # two hosted customers
    platform.pay(sender, recipient, 750)                        # on-ramp, transfer and off-ramp, one block each
    return platform, copy.deepcopy(platform.ledger.chain)       # the platform and an independent copy of its blocks


@test
def a_valid_chain_replays_to_the_right_state():
    """Replaying the chain rebuilds the balances, and the mZAR supply equals the balances added together."""
    platform, blocks = payment_chain()                          # a chain holding one full payment
    state, problem = replay(blocks, platform.ledger.genesis_hash)   # validate and rebuild the account table
    assert problem is None                                      # the chain is valid
    assert (state.minted, state.burned, state.supply) == (750, 750, 0)   # everything minted was later burned
    assert sum(state.balances.values()) == state.supply         # no mZAR exists outside the balances
    assert state.rewards[platform.validator.address] == 3 * BLOCK_REWARD   # three blocks, three rewards, paid in the native coin


@test
def an_edited_block_is_detected():
    """Changing a confirmed amount breaks the block's stored hash."""
    _, blocks = payment_chain()                                 # a valid chain
    blocks[2].transactions[1]["amount"] = 7500                  # raise the confirmed transfer from 750 to 7,500
    assert "self-hash pillar failed" in replay(blocks)[1]       # the stored hash no longer matches the contents


@test
def an_edited_block_that_is_mined_again_is_still_detected():
    """Redoing the proof of work does not help a forger, because the signature no longer matches."""
    _, blocks = payment_chain()                                 # a valid chain
    blocks[2].transactions[1]["amount"] = 7500                  # the same edit as before
    mine_again(blocks, 2)                                       # repair every hash and link from block 2 onwards
    assert "signature does not match" in replay(blocks)[1]      # the hashes are consistent, but the Sender never signed 7,500


@test
def a_miner_cannot_pay_itself_more_or_twice():
    """A larger reward, or a second reward in one block, invalidates the block even after it is mined again."""
    _, blocks = payment_chain()                                 # a valid chain
    inflated = copy.deepcopy(blocks)                            # a copy to damage
    inflated[1].transactions[0]["amount"] = 1_000_000           # the miner awards itself a million coins
    mine_again(inflated, 1)                                     # and redoes the proof of work
    assert "coinbase is not the reward the rules allow" in replay(inflated)[1]   # the reward is rebuilt and compared
    doubled = copy.deepcopy(blocks)                             # another copy to damage
    doubled[1].transactions.append(make_coinbase("a" * 40, BLOCK_REWARD, doubled[1].timestamp, 1))   # a second reward in block 1
    mine_again(doubled, 1)                                      # and redoes the proof of work
    assert "extra coinbase" in replay(doubled)[1]               # only one reward per block


@test
def an_under_mined_block_is_refused():
    """A block mined to a lower difficulty than the genesis block declares is refused."""
    platform = RemittancePlatform(difficulty=3, seed="tests")   # a network that demands three leading zeros
    platform.mine()                                             # one honest block
    blocks = copy.deepcopy(platform.ledger.chain)               # a copy to damage
    blocks[1].difficulty = 2                                    # the block now declares only two zeros
    mine_with_stats(blocks[1], 2)                               # and is mined to that easier target
    assert "below the network's 3" in replay(blocks)[1]         # the genesis block's difficulty still applies


@test
def the_charter_fixes_who_may_mint():
    """Swapping the reserve agent in the genesis block gives a different network, on which the old mints are invalid."""
    platform, blocks = payment_chain()                          # a valid chain
    mallory = Wallet.from_seed("tests/mallory")                 # an attacker who wants to become the reserve agent
    forged = Ledger(mallory.address, TEST_DIFFICULTY).chain[0]  # a genesis block naming the attacker instead
    assert forged.hash != platform.ledger.genesis_hash          # the genesis hash commits to the reserve agent
    swapped = [forged] + blocks[1:]                             # the attacker's genesis under the real blocks
    mine_again(swapped, 1)                                      # with every link and proof of work repaired
    assert "Genesis: does not match" in replay(swapped, platform.ledger.genesis_hash)[1]   # a verifier who knows the real genesis hash stops here
    assert "only the reserve agent" in replay(swapped)[1]       # and without it, the original mint is no longer authorised


@test
def the_fork_choice_rule_validates_before_comparing():
    """A longer valid chain is adopted, a longer invalid chain never is, and a chain of equal length changes nothing."""
    first = RemittancePlatform(difficulty=TEST_DIFFICULTY, seed="tests")    # one node
    second = RemittancePlatform(difficulty=TEST_DIFFICULTY, seed="tests")   # a second node with the same genesis block
    assert first.ledger.genesis_hash == second.ledger.genesis_hash         # the same seed gives the same reserve agent
    customer = first.register_wallet("Customer")["address"]                # a customer known to the first node
    first.on_ramp(customer, 500)                                           # a deposit and mint on the first node
    first.mine()                                                           # block 1 on the first node
    first.mine()                                                           # block 2 on the first node
    longer = copy.deepcopy(first.ledger.chain)                             # the first node's chain: three blocks
    assert second.ledger.consider_chain(longer) == (True, "adopted the longer valid chain")   # longer and valid, so adopted
    assert second.ledger.account(customer)["confirmed"] == 500             # the second node now sees the customer's balance
    assert second.ledger.consider_chain(longer)[0] is False                # the same chain again is not longer
    invalid = copy.deepcopy(longer)                                        # a copy to damage
    invalid[1].transactions[1]["amount"] = 5000                            # inflate the mint
    mine_again(invalid, 1)                                                 # repair the hashes
    invalid.append(copy.deepcopy(invalid[-1]))                             # make it longer than the honest chain
    invalid[-1].index += 1                                                 # the extra block takes the next position
    invalid[-1].transactions = [make_coinbase(first.validator.address, BLOCK_REWARD, invalid[-1].timestamp, invalid[-1].index)]   # with its own reward
    mine_again(invalid, len(invalid) - 1)                                  # link and mine the extra block
    adopted, message = second.ledger.consider_chain(invalid)               # offer the longer, invalid chain
    assert adopted is False and "invalid" in message                       # length cannot make up for a forged mint



Appending to tests.py


### 18.5 Group 4: the platform, and the test runner

In [55]:
%%writefile -a tests.py
# ===========================================================================
# GROUP 4: THE PLATFORM
# ===========================================================================

@test
def the_reserve_never_falls_below_the_supply():
    """At every step of a payment, including the moments between a request and its block, the reserve covers the supply."""
    platform, sender, recipient = new_platform()                # two hosted customers
    steps = [lambda: platform.on_ramp(sender, 1000), platform.mine,                 # deposit, then confirm the mint
             lambda: platform.transfer(sender, recipient, 750), platform.mine,      # transfer, then confirm it
             lambda: platform.off_ramp(recipient, 750), platform.mine]              # burn, then confirm it and pay out
    for action in steps:                                        # run the payment one step at a time
        action()                                                # carry out the step
        report = platform.reserve_report()                      # compare the cash book with the chain
        assert report["fully_backed"] and report["surplus"] >= 0    # the reserve rule holds after every single step
    assert (report["reserve_balance"], report["supply"]) == (250, 250)   # and ends exactly matched


@test
def a_receipt_is_signed_and_cannot_be_altered():
    """The reserve agent's signature covers the whole receipt."""
    platform, sender, recipient = new_platform()                # two hosted customers
    platform.pay(sender, recipient, 750)                        # a full payment, which issues one receipt
    receipt = platform.find_record("RCPT-0001")                 # the receipt
    agent = platform.reserve_agent.address                      # the address named in the genesis charter
    assert receipt_signature_valid(receipt, agent)              # the genuine receipt verifies
    assert not receipt_signature_valid({**receipt, "paid_out": 7500}, agent)        # a changed amount does not
    assert not receipt_signature_valid(receipt, recipient)      # and it only verifies against the reserve agent's address


@test
def the_platform_signs_only_for_keys_it_holds():
    """Hosted actions are refused for a self-custody wallet, and the on-ramp is refused for an unregistered address."""
    platform, sender, recipient = new_platform()                # two hosted customers
    own_key = Wallet.from_seed("tests/own-key")                 # a customer who keeps their own key
    platform.register_wallet("Own key", own_key.address)        # registered by address only
    assert refused(lambda: platform.transfer(own_key.address, recipient, 1), 403)   # the platform cannot sign a transfer for it
    assert refused(lambda: platform.off_ramp(own_key.address, 1), 403)              # nor a burn
    assert refused(lambda: platform.on_ramp("f" * 40, 100), 403)                    # an unregistered address cannot deposit
    assert refused(lambda: platform.register_wallet("Sender"), 409)                 # a name can be registered only once
    assert refused(lambda: platform.on_ramp(sender, 2.5), 400)                      # a decimal amount is refused before any money moves
    assert platform.reserve.balance == 0 and platform.ledger.pending == []          # none of the refusals changed anything


@test
def a_self_custody_customer_can_redeem():
    """A registered customer who holds their own key signs the burn themselves and is still paid out."""
    platform, ledger, alice, _ = funded_ledger()                # Alice keeps her own key and holds 100 mZAR
    platform.submit_signed(signed(alice, ledger, platform.reserve_agent.address, 30, kind="burn"))   # Alice signs her own burn
    receipt = platform.mine()["receipts_issued"][0]             # mining confirms it and triggers the payout
    assert (receipt["status"], receipt["paid_out"], platform.reserve.balance) == ("paid", 30, 70)   # paid, and the reserve fell by 30


@test
def an_unregistered_redemption_is_held():
    """mZAR burned by an address that was never registered is not paid out, and the reserve stays over-backed."""
    platform, ledger, alice, _ = funded_ledger()                # Alice holds 100 mZAR
    stranger = Wallet.from_seed("tests/stranger")               # an address the platform has never registered
    platform.submit_signed(signed(alice, ledger, stranger.address, 30))             # Alice pays the stranger 30 mZAR
    platform.mine()                                             # confirm the transfer
    platform.submit_signed(signed(stranger, ledger, platform.reserve_agent.address, 30, kind="burn"))   # the stranger burns it
    receipt = platform.mine()["receipts_issued"][0]             # mining confirms the burn
    assert receipt["status"].startswith("held") and receipt["paid_out"] == 0        # no payout to an unknown party
    assert platform.reserve_report()["surplus"] == 30           # the 30 rand stays in reserve, above the remaining supply


@test
def a_payout_waits_for_the_required_confirmations():
    """With two confirmations required, the block holding the burn pays nothing; the next block does."""
    platform, sender, recipient = new_platform(confirmations_required=2)   # a stricter finality rule
    platform.on_ramp(sender, 100)                               # a deposit
    platform.mine()                                             # block 1: the mint, with one confirmation
    platform.transfer(sender, recipient, 100)                   # accepted, because the pool applies the same rules as a block
    platform.off_ramp(recipient, 100)                           # the burn follows in the same pool
    assert platform.mine()["receipts_issued"] == []             # block 2 holds the burn but pays nothing yet
    assert platform.reserve.balance == 100                      # the money is still in reserve
    assert len(platform.mine()["receipts_issued"]) == 1         # block 3 gives the burn its second confirmation
    assert platform.reserve.balance == 0                        # and only now is the payout made


@test
def two_runs_with_the_same_seed_are_identical():
    """The whole chain, every identifier included, is the same on every run."""
    hashes = []                                                 # the block hashes of each run
    for _ in range(2):                                          # run the same payment twice
        platform, sender, recipient = new_platform()            # a fresh platform each time
        platform.pay(sender, recipient, 750)                    # the same payment
        hashes.append([block.hash for block in platform.ledger.chain])   # record every block hash
    assert hashes[0] == hashes[1]                               # identical, block for block

# ===========================================================================
# RUNNING THE TESTS
# ===========================================================================

def run_tests() -> int:
    """Run every registered test, print one line for each, and return how many failed."""
    failures = 0                                                # nothing has failed yet
    for function in TESTS:                                      # run the tests in the order they are written
        name = function.__name__.replace("_", " ")              # turn the function name into a readable sentence
        try:                                                    # an assert that fails raises an error
            function()                                          # run the test
            print(f"  [PASS] {name}")                           # every assert held
        except Exception as error:                              # an assert failed, or the test crashed
            failures += 1                                       # count the failure
            print(f"  [FAIL] {name}: {type(error).__name__} {error}")   # say which test and why
    print(f"\n{len(TESTS) - failures} passed, {failures} failed")   # the final count
    return failures                                             # the caller turns this into an exit status


if __name__ == "__main__":      # True only when this file is run directly, not when it is imported
    raise SystemExit(1 if run_tests() else 0)   # exit with an error status if any test failed


Appending to tests.py


In [56]:
import tests                        # the file written by the cells above
importlib.reload(tests)             # reload it in case it was written again
failures = tests.run_tests()        # run every test and print one line for each

  [PASS] assignment 2 block hash is unchanged
  [PASS] five field transaction is unchanged
  [PASS] seeded keys and signatures are repeatable
  [PASS] a signature covers every field
  [PASS] a valid transfer is accepted and moves value
  [PASS] only the reserve agent may mint
  [PASS] nobody can spend more than they hold
  [PASS] an altered transaction is refused
  [PASS] a transaction cannot be replayed
  [PASS] sequence numbers must be used in order
  [PASS] two pending spends cannot overdraw
  [PASS] malformed transactions are refused
  [PASS] a valid chain replays to the right state
  [PASS] an edited block is detected
  [PASS] an edited block that is mined again is still detected
  [PASS] a miner cannot pay itself more or twice
  [PASS] an under mined block is refused
  [PASS] the charter fixes who may mint
  [PASS] the fork choice rule validates before comparing
  [PASS] the reserve never falls below the supply
  [PASS] a receipt is signed and cannot be altered
  [PASS] the platf

  [PASS] an altered transaction is refused


  [PASS] a transaction cannot be replayed


  [PASS] sequence numbers must be used in order
  [PASS] two pending spends cannot overdraw
  [PASS] malformed transactions are refused
  [PASS] a valid chain replays to the right state


  [PASS] an edited block is detected


  [PASS] an edited block that is mined again is still detected
  [PASS] a miner cannot pay itself more or twice
  [PASS] an under mined block is refused
  [PASS] the charter fixes who may mint
  [PASS] the fork choice rule validates before comparing


  [PASS] the reserve never falls below the supply
  [PASS] a receipt is signed and cannot be altered
  [PASS] the platform signs only for keys it holds
  [PASS] a self custody customer can redeem
  [PASS] an unregistered redemption is held
  [PASS] a payout waits for the required confirmations


  [PASS] two runs with the same seed are identical

26 passed, 0 failed


The node keeps running until the kernel is restarted, so the explorer stays available at the address printed in section 8, followed by `/explorer`. Running `stop_background_nodes()` in a new cell stops it sooner.